# Sample representation via sampleCLR

In [ ]:
import sys
from pathlib import Path
import sampleclr
print(sampleclr.__file__) 

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import os

In [ ]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 300)

## Load data

In [ ]:
os.getcwd()

In [ ]:
os.chdir('SET WORKING DIRECTORY')

In [ ]:
adata_taurus = sc.read_h5ad("data/taurus_data/taurus_hgca_full_mapped_hvg.h5ad")

In [ ]:
adata_taurus

In [ ]:
adata_taurus.obs['Disease'].value_counts()

In [ ]:
adata_taurus.obs['Inflammation'].value_counts()

In [ ]:
adata_taurus_CD = adata_taurus[adata_taurus.obs['Disease']!="UC"].copy()

In [ ]:
adata_taurus_CD.obs['Inflammation_binary'] = adata_taurus_CD.obs['Inflammation'].replace({'Healthy':'Non_Inflamed'})

## SampleCLR pipeline across lineages (taurus CD only)

One sample-level embedding per `sample_id` from the shared 30-dim scANVI latent, compared against patpy baselines.

In [ ]:
# pipeline imports
import torch
import scipy.sparse as sp
from scipy.spatial.distance import pdist, squareform
from sklearn.model_selection import train_test_split

import patpy as pat
import ehrapy as ehr
from sampleclr.utils import get_sample_representations_from_adata

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# what to analyse: the 30-dim scANVI latent shared by reference and query
adata         = adata_taurus_CD
LAYER         = "X_scANVI_across_lin"
SAMPLE_KEY    = "sample_id"
CELLGROUP_KEY = "predictions"

assert LAYER in adata.obsm, f"{LAYER!r} not in obsm {list(adata.obsm)}"
print("device:", device, "| cells:", adata.n_obs, "| latent:", adata.obsm[LAYER].shape)

In [ ]:
adata_taurus_CD

In [ ]:
# how many samples survive each cell-count threshold
counts = adata.obs[SAMPLE_KEY].value_counts()
print("samples retained by threshold:")
for t in [20, 50, 100, 200, 300, 500]:
    print(f"  >= {t:4d} cells: {(counts >= t).sum():4d} / {len(counts)}")

In [ ]:
MIN_CELLS = 300
keep_samples = counts[counts >= MIN_CELLS].index
mask = adata.obs[SAMPLE_KEY].isin(keep_samples).to_numpy()
print(f"\nMIN_CELLS={MIN_CELLS}: {len(keep_samples)} samples, {mask.sum()} cells")

In [ ]:
# cell-level object for patpy / sampleCLR; X is empty because every method reads obsm[LAYER]
obs = adata.obs.loc[mask].copy()
emb = np.asarray(adata.obsm[LAYER][mask])
adata_pb = ad.AnnData(
    X=sp.csr_matrix((emb.shape[0], 0), dtype=np.float32),
    obs=obs, var=pd.DataFrame(index=[]), obsm={LAYER: emb},
)

In [ ]:
adata_taurus_CD.obs.columns

In [ ]:

# one row per sample: the metadata table everything attaches to
meta = (obs.drop_duplicates(subset=[SAMPLE_KEY]).set_index(SAMPLE_KEY)
        .loc[keep_samples, ['Patient', 'Disease', 'Site', 'Treatment',
       'Disease_duration', 'Inflammation','Remission_status', 'Age', 'Gender',
       'Inflammation_score','Inflammation_binary', 'Ileum_vs_Colon', 'LibraryType',
       'Batch']])
meta["n_cells"] = counts.loc[keep_samples]
meta_adata = ad.AnnData(obs=meta)
meta_adata.uns["sample_representations"] = []
print(meta_adata)
meta.head()

In [ ]:
def store_rep(name, reps, dist=None, samples=None):
    """Attach a sample representation (+cosine distances if none given) to meta_adata."""
    order = list(meta_adata.obs_names)
    if isinstance(reps, pd.DataFrame):
        reps_arr = reps.loc[order].to_numpy()
    else:
        samples = list(samples)
        pos = {s: i for i, s in enumerate(samples)}
        reps_arr = np.asarray(reps)[[pos[s] for s in order]]
    if dist is None:
        dist_df = pd.DataFrame(squareform(pdist(reps_arr, metric="cosine")), index=order, columns=order)
    elif isinstance(dist, pd.DataFrame):
        dist_df = dist.loc[order, order]
    else:
        dist_df = pd.DataFrame(np.asarray(dist), index=list(samples), columns=list(samples)).loc[order, order]
    meta_adata.obsm[f"{name}_representations"] = reps_arr
    meta_adata.obsm[f"{name}_distances"] = dist_df
    lst = meta_adata.uns.setdefault("sample_representations", [])
    if name not in lst:
        lst.append(name)
    print(f"stored {name!r}: reps {reps_arr.shape}")

### Baseline sample representations (patpy)

In [ ]:
# baseline 1: pseudobulk (mean scANVI latent per sample)
pb = pat.tl.Pseudobulk(sample_key=SAMPLE_KEY, cell_group_key=CELLGROUP_KEY, layer=LAYER)
pb.prepare_anndata(adata_pb)
pb_dist = pb.calculate_distance_matrix(force=True)

In [ ]:
store_rep("pseudobulk", pb.sample_representation, pb_dist, pb.samples)

In [ ]:
# baseline 2: cell-type composition (CLR of cell-type fractions)
comp = pat.tl.CellGroupComposition(sample_key=SAMPLE_KEY, cell_group_key=CELLGROUP_KEY, layer=LAYER)
comp.prepare_anndata(adata_pb)
comp_dist = comp.calculate_distance_matrix(force=True)

In [ ]:
store_rep("composition", comp.sample_representation, comp_dist, comp.samples)

In [ ]:
# baseline 3: GloScope (kNN density divergence); use_gpu=False if RAPIDS and torch disagree on CUDA
try:
    glo = pat.tl.GloScope_py(sample_key=SAMPLE_KEY, layer=LAYER, k=30, use_gpu=True)
    glo.prepare_anndata(adata_pb)
    glo_dist = glo.calculate_distance_matrix(force=True)
    store_rep("gloscope", glo.sample_representation, glo_dist, glo.samples)
except Exception as e:
    print("GloScope skipped:", repr(e))

In [ ]:
# baseline 4: cell-type pseudobulk (optional)
# ct_pb = pat.tl.GroupedPseudobulk(sample_key=SAMPLE_KEY, cell_group_key=CELLGROUP_KEY, layer=LAYER)
# ct_pb.prepare_anndata(adata_pb)
# ct_pb_dist = ct_pb.calculate_distance_matrix(force=True)

In [ ]:
# ct_pb.sample_representation

In [ ]:
# store_rep("CT_pseudobulk", ct_pb.sample_representation, ct_pb_dist, ct_pb.samples)

## Signal beyond Batch

Remission is constant within patient (19 patients, 16 labelled) and `Batch` / `LibraryType` are confounded with it, so every split below is over patients and the batch-corrected and uncorrected numbers are read as a pair.

In [ ]:
# setup + audit: how much of remission is already explained by technical covariates
import json
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SCLR_REPO  = Path("/ictstr01/groups/ml01/workspace/christopher.lance/SampleCLR")
OUTPUT_DIR = Path("hca-gut-atlas-downstream/data/taurus_data/sampleclr_beyond_batch")
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "representations").mkdir(parents=True, exist_ok=True)

PATIENT_KEY = "Patient"
BATCH_KEY   = "Batch"
TARGET      = "Remission_clean"      # created in the next cell

# colour follows the family of a representation, never its rank
FAMILY_COLORS = {
    "baseline":       "#2a78d6",
    "zero-shot":      "#eb6834",
    "supervised":     "#1baf7a",
    "batch-corrected": "#4a3aa7",
}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

sm = meta_adata.obs
print(f"{len(sm)} samples | {sm[PATIENT_KEY].nunique()} patients | "
      f"median {sm[PATIENT_KEY].value_counts().median():.0f} samples/patient\n")

# 1. remission is a patient-level label
lab = sm["Remission_status"].astype(str).str.strip()
print(pd.DataFrame({
    "samples":  lab.value_counts(),
    "patients": sm.assign(_l=lab).groupby("_l", observed=True)[PATIENT_KEY].nunique(),
}).to_string(), "\n")

n_multi = sm.assign(_l=lab).groupby(PATIENT_KEY, observed=True)["_l"].nunique().gt(1).sum()
print(f"patients with more than one remission value: {int(n_multi)}"
      "  <- 0 means the label carries no within-patient information\n")

# 2. how much of the outcome is already in the technical covariates?
def majority_rule_accuracy(feature, target):
    """Accuracy of 'predict the majority target class within each level of feature'."""
    df = pd.DataFrame({"f": feature.astype(str), "t": target.astype(str)}).dropna()
    pred = df.groupby("f", observed=True)["t"].transform(lambda s: s.mode().iat[0])
    return (pred == df["t"]).mean()

cd = lab.isin(["Remission", "Non_Remission"]).to_numpy()      # drop healthy
print(f"majority-class baseline                 : {lab[cd].value_counts(normalize=True).max():.3f}")
for c in [BATCH_KEY, "LibraryType", "Site", "Ileum_vs_Colon", PATIENT_KEY]:
    print(f"predict remission from {c:<17}: {majority_rule_accuracy(sm.loc[cd, c], lab[cd]):.3f}")

print("\n--- Batch x Remission (samples) ---")
print(pd.crosstab(sm.loc[cd, BATCH_KEY].astype(str), lab[cd]).to_string())
print("\n--- LibraryType x Remission (samples) ---")
print(pd.crosstab(sm.loc[cd, "LibraryType"].astype(str), lab[cd]).to_string())
print("\n--- patients per Batch (Batch is nested inside Patient) ---")
print(sm.groupby(BATCH_KEY, observed=True)[PATIENT_KEY].nunique().to_string())

#### (a) Clean the target

The "no label" level is the literal string `'None '`; mapping it to `NaN` keeps healthy samples in the contrastive loss but out of the classification head.

In [ ]:
# (a) clean the target: 'None ' / 'Not_avail' -> NaN
REMISSION_NA = {"None", "Not_avail", "nan", "NaN", "<NA>", ""}

def clean_remission(s):
    s = s.astype(str).str.strip()
    return s.where(~s.isin(REMISSION_NA), np.nan)

for _ad in (adata_pb, meta_adata):
    _ad.obs[TARGET] = clean_remission(_ad.obs["Remission_status"])
    # drop any stale encoding so SampleCLR re-encodes from the cleaned column
    _ad.obs.drop(columns=[f"{TARGET}_encoded"], errors="ignore", inplace=True)

print("cells   :", adata_pb.obs[TARGET].value_counts(dropna=False).to_dict())
print("samples :", meta_adata.obs[TARGET].value_counts(dropna=False).to_dict())
print("labelled patients:",
      meta_adata.obs.loc[meta_adata.obs[TARGET].notna(), PATIENT_KEY].nunique(),
      "of", meta_adata.obs[PATIENT_KEY].nunique())

#### (b) Patient-grouped train / val / test

Splits are drawn over patients and expanded to sample IDs; `test_ids` is a held-back reserve, the real scoring is the leave-one-patient-out pass below.

In [ ]:
# (b) patient-grouped splits, expanded to sample IDs
def patient_split(seed=0, test_frac=0.25, val_frac=0.25, verbose=True):
    """Split patients (not samples) into train/val/test; returns three lists of sample IDs."""
    obs = meta_adata.obs
    lab_by_pat = (obs.loc[obs[TARGET].notna()]
                     .groupby(PATIENT_KEY, observed=True)[TARGET].first())
    pats, y = lab_by_pat.index.to_numpy(), lab_by_pat.to_numpy()

    rest_i, test_i = next(StratifiedShuffleSplit(
        n_splits=1, test_size=test_frac, random_state=seed).split(pats, y))
    tr_i, val_i = next(StratifiedShuffleSplit(
        n_splits=1, test_size=val_frac, random_state=seed).split(pats[rest_i], y[rest_i]))

    test_pats  = set(pats[test_i])
    val_pats   = set(pats[rest_i][val_i])
    train_pats = (set(pats[rest_i][tr_i])
                  | set(obs.loc[obs[TARGET].isna(), PATIENT_KEY].astype(str).unique()))

    # a patient must live in exactly one split -- this is the leakage guard
    assert not (train_pats & val_pats), train_pats & val_pats
    assert not (train_pats & test_pats), train_pats & test_pats
    assert not (val_pats & test_pats),   val_pats & test_pats

    def ids(p):
        return obs.index[obs[PATIENT_KEY].astype(str).isin({str(x) for x in p})].tolist()

    splits = {"train": (train_pats, ids(train_pats)),
              "val":   (val_pats,   ids(val_pats)),
              "test":  (test_pats,  ids(test_pats))}
    if verbose:
        for name, (p, s) in splits.items():
            print(f"  {name:5s}: {len(p):2d} patients, {len(s):3d} samples  "
                  f"{obs.loc[s, TARGET].value_counts().to_dict()}")
    return [splits[k][1] for k in ("train", "val", "test")]

train_ids, val_ids, test_ids = patient_split(seed=0)
assert not (set(train_ids) & set(val_ids) & set(test_ids))
print(f"\n{len(train_ids) + len(val_ids) + len(test_ids)} of {meta_adata.n_obs} samples assigned")

#### (c) Model config + ablation grid

Grid over supervision (none / remission / inflammation), batch-aware sampling and `lambda_`, plus derived arms with `Inflammation_score` projected out post-hoc.

In [ ]:
# (c) model config + ablation grid
SCLR_SIZE = "tiny"                       # "tiny" | "medium" | "large"
sclr_cfg  = json.loads((SCLR_REPO / f"configs/models/{SCLR_SIZE}.json").read_text())
sclr_cfg["contrastive_loss_temperature"] = 0.1
sclr_cfg.pop("batch_size", None)                      # passed explicitly below
print(f"{SCLR_SIZE}.json ->", json.dumps(sclr_cfg, indent=2))

CELL_SELECTION   = "random"              # 'eigenvector' needs cugraph
BASE_TASKS       = {"classification": [TARGET]}
INFLAM_KEY       = "Inflammation_score"  # continuous histology score, one value per sample
INFLAM_TASKS     = {"regression": [INFLAM_KEY]}
LAMBDA_DEFAULT   = 1.0
RUN_LAMBDA_SWEEP = True
LAMBDA_SWEEP     = [0.5, 1.0, 3.0, 10.0]
N_SEEDS          = 1                     # raise to average over split draws

CONFIGS = [
    dict(rep_name="sclr_zeroshot",             tasks=None,         batch_aware=False, lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_zeroshot_batchaware",  tasks=None,         batch_aware=True,  lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_remission",            tasks=BASE_TASKS,   batch_aware=False, lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_remission_batchaware", tasks=BASE_TASKS,   batch_aware=True,  lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_remission_batchcorr",  batch_aware=False,  lambda_=LAMBDA_DEFAULT,
         tasks={**BASE_TASKS, "batch_correction": BATCH_KEY}),
    # inflammation severity alone: a biopsy-level, within-patient axis, orthogonal to remission
    dict(rep_name="sclr_inflammation",         tasks=INFLAM_TASKS, batch_aware=False, lambda_=LAMBDA_DEFAULT),
]
if RUN_LAMBDA_SWEEP:
    CONFIGS += [dict(rep_name=f"sclr_remission_lam{l:g}".replace(".", "p"),
                     tasks=BASE_TASKS, batch_aware=False, lambda_=l)
                for l in LAMBDA_SWEEP if l != LAMBDA_DEFAULT]

# derived arms: Inflammation_score projected out post-hoc, baselines included as the reference
RESIDUALIZE_ON = INFLAM_KEY
RESIDUALIZE    = ["pseudobulk", "composition",
                  "sclr_remission", "sclr_remission_batchaware", "sclr_remission_batchcorr"]
RESID_SUFFIX   = "_inflamresid"

# residualisation is a secondary encoding (marker shape / hatch); an arm keeps its parent's colour
def is_residualised(name):
    return RESID_SUFFIX in name

def rep_family(name):
    n = name.lower().replace(RESID_SUFFIX, "")          # residualised keeps its parent's family
    if not n.startswith("sclr"):                        return "baseline"
    if "batchcorr" in n or "batchaware" in n:           return "batch-corrected"
    if "zeroshot" in n:                                 return "zero-shot"
    return "supervised"

print(f"\n{len(CONFIGS)} configs x {N_SEEDS} seed(s) = {len(CONFIGS) * N_SEEDS} models")
for c in CONFIGS:
    print(f"  {c['rep_name']:34s} tasks={str(c['tasks']):55s} "
          f"batch_aware={c['batch_aware']!s:5s} lambda={c['lambda_']}")
print(f"\n+ {len(RESIDUALIZE)} derived arms (no training): {RESIDUALIZE_ON} projected out of")
for b in RESIDUALIZE:
    print(f"  {b}{RESID_SUFFIX}")

In [ ]:
# train the grid, skipping anything already cached on disk
import contextlib, io, hashlib

subset_size = int(min(counts.loc[keep_samples].max(), 512))
bb_reps = []
splits_by_seed = {}          # seed -> train sample IDs, so residualisation uses the right split

def cfg_fingerprint(cfg, seed):
    """Hash of everything that changes the model, so a stale cached representation is caught."""
    payload = json.dumps({"cfg": cfg, "seed": seed, "sclr_cfg": sclr_cfg,
                          "layer": LAYER, "min_cells": MIN_CELLS},
                         sort_keys=True, default=str)
    return hashlib.sha1(payload.encode()).hexdigest()[:12]

for seed in range(N_SEEDS):
    tr, va, te = patient_split(seed=seed, verbose=(seed == 0))
    splits_by_seed[seed] = tr

    for cfg in CONFIGS:
        rep_name  = f"{cfg['rep_name']}_seed{seed}"
        rep_path  = OUTPUT_DIR / "representations" / f"{rep_name}_representations.npy"
        ckpt_path = OUTPUT_DIR / "models" / f"{rep_name}.pt"
        fingerprint = cfg_fingerprint(cfg, seed)

        if rep_path.exists():
            cached_fp = (torch.load(ckpt_path, map_location="cpu", weights_only=False).get("fingerprint")
                         if ckpt_path.exists() else None)
            if cached_fp is not None and cached_fp != fingerprint:
                print(f"!! {rep_name}: cached representation was trained under a different config "
                      f"({cached_fp} != {fingerprint}). Delete {rep_path.name} to retrain; "
                      f"loading the stale one for now.")
            store_rep(rep_name, np.load(rep_path), dist=None, samples=list(meta_adata.obs_names))
            bb_reps.append(rep_name)
            continue

        print(f"\n{'=' * 78}\n{rep_name}  |  tasks={cfg['tasks']}  "
              f"batch_aware={cfg['batch_aware']}  lambda_={cfg['lambda_']}\n{'=' * 78}")

        kw = dict(
            adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
            train_ids=tr, val_ids=va, test_ids=te,
            supervised_class_balancing="inverse",     # 68 Remission vs 28 Non_Remission samples
            lambda_=cfg["lambda_"],
            n_cells_per_sample=[50, 150], batch_size=32,
            early_stopping_patience=50,
            num_warmup_epochs_stage1=10, num_warmup_epochs_stage2=10,
            device=device, seed=seed, verbose=False, **sclr_cfg,
        )
        if cfg["tasks"] is not None:
            kw["tasks"] = cfg["tasks"]
        if cfg["batch_aware"]:
            kw.update(use_batch_aware_sampler=True,
                      batch_sampler_batch_col=BATCH_KEY,
                      # Batch_4 has only 5 samples -- merge undersized batches for the sampler
                      batch_sampler_pseudo_batch_strategy="similarity")

        model = sampleclr.ContrastiveModel(**kw)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            model.train(num_epochs_stage1=100, num_epochs_stage2=150,
                        stage1_val_metric="loss",
                        stage2_val_metric="total" if cfg["tasks"] else "loss",
                        verbose=False)

        reps_arr = get_sample_representations_from_adata(
            projector=model.projector, aggregator=model.aggregator,
            adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
            meta_obs_names=list(meta_adata.obs_names), subset_size=subset_size,
            device=device, cell_selection=CELL_SELECTION)

        store_rep(rep_name, reps_arr, dist=None, samples=list(meta_adata.obs_names))
        bb_reps.append(rep_name)

        np.save(rep_path, reps_arr)
        torch.save({
            "projector":  model.projector.state_dict(),
            "aggregator": model.aggregator.state_dict(),
            # every head group is saved, so load_sclr_model can restore any arm
            "classifiers":        [c.state_dict() for c in getattr(model, "classifiers", [])],
            "regressors":         [r.state_dict() for r in getattr(model, "regressors", [])],
            "ordinal_regressors": [o.state_dict() for o in getattr(model, "ordinal_regressors", [])],
            "cfg": cfg, "seed": seed, "layer": model.layer, "output_dim": model.output_dim,
            "sclr_cfg": sclr_cfg, "fingerprint": fingerprint,
            "split": {"train": tr, "val": va, "test": te},
        }, ckpt_path)

meta_adata.obs["split"] = "train"
meta_adata.obs.loc[val_ids,  "split"] = "val"
meta_adata.obs.loc[test_ids, "split"] = "test"
print("\ntrained/loaded:", bb_reps)

# derived arms: project Inflammation_score out of an existing representation
def residualise(X, values, fit_mask):
    """OLS removal of the linear `values` component, with coefficients fit on train patients only."""
    X = np.asarray(X, dtype=np.float64)
    v = np.asarray(values, dtype=np.float64)
    D = np.column_stack([np.ones(len(v)), v])
    ok = fit_mask & np.isfinite(v) & np.isfinite(X).all(axis=1)
    coef, *_ = np.linalg.lstsq(D[ok], X[ok], rcond=None)
    return (X - D @ coef).astype(np.float32)

_score = pd.to_numeric(meta_adata.obs[RESIDUALIZE_ON], errors="coerce").to_numpy(float)
assert np.isfinite(_score).all(), \
    f"{RESIDUALIZE_ON} has {int(np.isnan(_score).sum())} NaN; the projection would be undefined there"
print(f"\nprojecting out {RESIDUALIZE_ON} (coefficients fit on train patients, applied to all "
      f"{len(_score)} samples)")

resid_reps = []
for base in RESIDUALIZE:
    # a seedless name means a patpy baseline (one copy); an sclr arm has one per seed
    names = ([base] if f"{base}_representations" in meta_adata.obsm
             else [f"{base}_seed{s}" for s in range(N_SEEDS)])
    for name in names:
        if f"{name}_representations" not in meta_adata.obsm:
            print(f"  skip {name}: no representation stored")
            continue
        # baselines carry no seed, so they are residualised on the seed-0 split
        seed = int(name.rsplit("_seed", 1)[1]) if "_seed" in name else 0
        fit  = meta_adata.obs_names.isin(splits_by_seed.get(seed, train_ids))
        out  = f"{name}{RESID_SUFFIX}"
        store_rep(out, residualise(meta_adata.obsm[f"{name}_representations"], _score, fit),
                  dist=None, samples=list(meta_adata.obs_names))
        resid_reps.append(out)

bb_reps += resid_reps
print("\nderived:", resid_reps)

#### (d) Benchmark, with `Patient` as the leakage detector

kNN recovery per covariate (`f1_macro_calibrated`: 0 = chance, 1 = perfect), with `Patient` scored uninverted so remission recovery that is only patient re-identification stays visible.

In [ ]:
# (d) benchmark: covariate recovery, with Patient as the leakage detector
bb_baselines = [m for m in ["pseudobulk", "composition", "gloscope", "CT_pseudobulk"]
                if f"{m}_representations" in meta_adata.obsm]
bb_methods   = bb_baselines + bb_reps
print("scoring:", bb_methods)

benchmark_schema_bb = {
    "relevant": {
        TARGET:         "classification",
        "Treatment":    "classification",
        "Inflammation": "classification",
        INFLAM_KEY:     "regression",
    },
    "technical": {                          # reported inverted: higher = less leakage
        PATIENT_KEY:   "classification",
        BATCH_KEY:     "classification",
        "LibraryType": "classification",
        "n_cells":     "regression",
    },
    "contextual": {
        "Ileum_vs_Colon": "classification",
        "Site":           "classification",
    },
}

# same covariates under "relevant" -> RAW recovery, which is what a diagnostic needs
leak_schema = {"relevant": {PATIENT_KEY:   "classification",
                            BATCH_KEY:     "classification",
                            "LibraryType": "classification",
                            TARGET:        "classification",
                            INFLAM_KEY:    "regression"}}

bb_scores = pd.concat(
    [pat.tl.evaluation.knn_prediction_score(meta_adata, benchmark_schema_bb, representations=[m])
     for m in bb_methods], ignore_index=True)
bb_leak = pd.concat(
    [pat.tl.evaluation.knn_prediction_score(meta_adata, leak_schema, representations=[m])
     for m in bb_methods], ignore_index=True)

bb_wide   = bb_scores.pivot_table(index="representation", columns="covariate", values="score")
leak_wide = bb_leak.pivot_table(index="representation", columns="covariate", values="score")
leak_wide = leak_wide.loc[bb_methods]        # keep grid order, not alphabetical

print("\n=== uninverted kNN recovery, f1_macro_calibrated: 0 = chance, 1 = perfect ===")
display(leak_wide[[TARGET, PATIENT_KEY, BATCH_KEY, "LibraryType"]]
        .rename(columns={TARGET: "Remission"})
        .style.format("{:.3f}").background_gradient(cmap="PiYG", vmin=0, vmax=1))

# did the projection remove Inflammation_score, and what did it cost? (regression -> signed spearman r)
_pairs = [(b, f"{b}{RESID_SUFFIX}") for b in leak_wide.index if f"{b}{RESID_SUFFIX}" in leak_wide.index]
if _pairs:
    removal = pd.DataFrame(
        [{"representation":        base,
          f"{INFLAM_KEY} (before)": leak_wide.loc[base, INFLAM_KEY],
          f"{INFLAM_KEY} (after)":  leak_wide.loc[res,  INFLAM_KEY],
          "Remission (before)":     leak_wide.loc[base, TARGET],
          "Remission (after)":      leak_wide.loc[res,  TARGET],
          PATIENT_KEY:              leak_wide.loc[res,  PATIENT_KEY]}
         for base, res in _pairs]).set_index("representation")
    removal["Remission cost"] = removal["Remission (after)"] - removal["Remission (before)"]

    print(f"\n=== {RESIDUALIZE_ON} removal check ===")
    print("left pair : spearman_r. Should collapse in magnitude. A small negative value is")
    print("            expected, not a bug -- the projection is fit on train patients only,")
    print("            so held-out samples end up slightly over-corrected.")
    print("right pair: f1_macro_calibrated, the price paid in remission recovery.")
    display(removal.style.format("{:+.3f}")
            .background_gradient(cmap="PuOr", subset=[f"{INFLAM_KEY} (before)",
                                                      f"{INFLAM_KEY} (after)"], vmin=-1, vmax=1)
            .background_gradient(cmap="PiYG", subset=["Remission cost"], vmin=-0.5, vmax=0.5))
else:
    print(f"\nno {RESID_SUFFIX} arms stored -- run the training cell first")

In [ ]:
%matplotlib inline
# does the Remission score track patient leakage? signal lives above the diagonal
fig, ax = plt.subplots(figsize=(7.2, 6.4))
ax.set_facecolor("none")

lim = max(0.35, leak_wide[[TARGET, PATIENT_KEY]].to_numpy().max() * 1.25)
ax.plot([0, lim], [0, lim], color=GRID, lw=2, zorder=1)
ax.annotate("remission = patient identity", xy=(lim * 0.62, lim * 0.62),
            xytext=(6, -14), textcoords="offset points", color=MUTED, fontsize=9)

# connector from each base arm to its residualised twin
for base in leak_wide.index:
    res = f"{base}{RESID_SUFFIX}"
    if res in leak_wide.index:
        ax.plot(leak_wide.loc[[base, res], PATIENT_KEY], leak_wide.loc[[base, res], TARGET],
                color=FAMILY_COLORS[rep_family(base)], lw=1.2, alpha=0.55, zorder=2)

seen = set()
for rep in leak_wide.index:
    fam  = rep_family(rep)
    resid = is_residualised(rep)
    x, y = leak_wide.loc[rep, PATIENT_KEY], leak_wide.loc[rep, TARGET]
    ax.scatter(x, y, s=110, color=FAMILY_COLORS[fam], marker="D" if resid else "o",
               edgecolor="white", linewidth=2, zorder=3, label=fam if fam not in seen else None)
    seen.add(fam)
    ax.annotate(rep.replace("_seed0", "").replace("sclr_", "").replace(RESID_SUFFIX, " -infl"),
                xy=(x, y), xytext=(8, -3), textcoords="offset points", fontsize=9, color=INK)

ax.set_xlabel("Patient recovery  (f1_macro_calibrated, 0 = chance)", color=INK)
ax.set_ylabel("Remission recovery  (f1_macro_calibrated, 0 = chance)", color=INK)
ax.set_title("Remission signal vs patient leakage\nabove the diagonal = beyond patient identity",
             color=INK, loc="left")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.grid(True, color=GRID, lw=0.8, zorder=0)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color("#c3c2b7")
ax.tick_params(colors=MUTED)

# two legends: colour = family, shape = whether Inflammation_score was projected out
leg_fam = ax.legend(frameon=False, loc="upper right", labelcolor=INK)
ax.add_artist(leg_fam)
shape_handles = [plt.Line2D([], [], marker=m, ls="", ms=9, color=MUTED, mec="white", mew=1.5, label=l)
                 for m, l in [("o", "as trained"), ("D", f"{RESIDUALIZE_ON} projected out")]]
ax.legend(handles=shape_handles, frameon=False, loc="lower right", labelcolor=INK, fontsize=9)
plt.tight_layout(); plt.show()

#### (e) Patient-level leave-one-patient-out evaluation

Sample embeddings pooled per patient, then leave-one-patient-out logistic regression; one-hot `Batch` goes through the same loop as the reference to beat.

In [ ]:
# (e) leave-one-patient-out at the patient level
_lab_ok = meta_adata.obs[TARGET].notna().to_numpy()
_pat    = meta_adata.obs[PATIENT_KEY].astype(str).to_numpy()[_lab_ok]
_y      = (meta_adata.obs[TARGET].to_numpy()[_lab_ok] == "Remission").astype(int)

y_pat = pd.Series(_y, index=_pat).groupby(level=0).first()
print(f"{len(y_pat)} labelled patients: "
      f"{int(y_pat.sum())} Remission / {int((1 - y_pat).sum())} Non_Remission")

def to_patient_matrix(X):
    """Mean-pool sample embeddings within each patient; rows aligned to y_pat."""
    return pd.DataFrame(np.asarray(X)[_lab_ok], index=_pat).groupby(level=0).mean().loc[y_pat.index]

def lopo_scores(Xp, yp, C=1.0):
    """Leave-one-patient-out CV; returns (auc, balanced_accuracy) over pooled predictions."""
    prob = np.full(len(Xp), np.nan)
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    for tr, te in LeaveOneGroupOut().split(Xp, yp, groups=np.asarray(Xp.index)):
        clf.fit(Xp.iloc[tr], yp.iloc[tr])
        prob[te] = clf.predict_proba(Xp.iloc[te])[:, 1]
    return roc_auc_score(yp, prob), balanced_accuracy_score(yp, (prob >= 0.5).astype(int))

rows = []
for m in bb_methods:
    auc, bacc = lopo_scores(to_patient_matrix(meta_adata.obsm[f"{m}_representations"]), y_pat)
    rows.append({"representation": m, "family": rep_family(m), "AUC": auc, "balanced_acc": bacc})

# technical-covariate references, through the identical LOPO loop
for col in [BATCH_KEY, "LibraryType"]:
    oh = pd.get_dummies(meta_adata.obs[col].astype(str)).to_numpy(dtype=float)
    auc, bacc = lopo_scores(to_patient_matrix(oh), y_pat)
    rows.append({"representation": f"reference: {col} (one-hot)", "family": "reference",
                 "AUC": auc, "balanced_acc": bacc})
rows.append({"representation": "reference: chance", "family": "reference",
             "AUC": 0.5, "balanced_acc": 0.5})

lopo = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
BATCH_REF = float(lopo.loc[lopo["representation"] == f"reference: {BATCH_KEY} (one-hot)", "AUC"].iat[0])
display(lopo.style.format({"AUC": "{:.3f}", "balanced_acc": "{:.3f}"})
        .background_gradient(cmap="PiYG", subset=["AUC"], vmin=0.3, vmax=1.0))
print(f"\nreference to beat -- {BATCH_KEY} alone: AUC {BATCH_REF:.3f}")
print("clears it:", lopo.loc[(lopo.family != "reference") & (lopo.AUC > BATCH_REF),
                             "representation"].tolist() or "none")

In [ ]:
%matplotlib inline
import matplotlib.patches as mpatches
# patient-level LOPO AUC against the technical references (hatched = Inflammation_score removed)
FAM_COLORS_ALL = {**FAMILY_COLORS, "reference": MUTED}

plot_df = lopo.sort_values("AUC")            # ascending -> best on top of an hbar
fig, ax = plt.subplots(figsize=(8.4, 0.42 * len(plot_df) + 1.8))
ax.set_facecolor("none")

bars = ax.barh(plot_df["representation"].str.replace("_seed0", "", regex=False),
               plot_df["AUC"],
               color=[FAM_COLORS_ALL[f] for f in plot_df["family"]],
               hatch=["//" if is_residualised(r) else "" for r in plot_df["representation"]],
               edgecolor="white", linewidth=0,
               height=0.62, zorder=3)

ax.set_ylim(-0.6, len(plot_df) - 0.5 + 0.95)
ax.axvline(0.5, color=GRID, lw=2, zorder=1)
ax.axvline(BATCH_REF, color=MUTED, lw=2, ls=(0, (4, 3)), zorder=2)
ax.annotate(f"{BATCH_KEY} alone ({BATCH_REF:.2f})", xy=(BATCH_REF, len(plot_df) - 0.1),
            xytext=(5, 0), textcoords="offset points",
            color=MUTED, fontsize=9, va="center")

for b, v in zip(bars, plot_df["AUC"]):
    ax.text(v + 0.008, b.get_y() + b.get_height() / 2, f"{v:.2f}",
            va="center", fontsize=9, color=INK)

ax.set_xlim(0, 1.06)
ax.set_xlabel("Leave-one-patient-out AUC  (16 patients)", color=INK)
ax.set_title("Remission prediction from patient-pooled sample embeddings",
             color=INK, loc="left")
ax.grid(True, axis="x", color=GRID, lw=0.8, zorder=0)
ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(colors=MUTED)
ax.tick_params(axis="y", length=0, labelcolor=INK)

handles = [plt.Line2D([], [], marker="s", ls="", ms=9, color=c, label=f)
           for f, c in FAM_COLORS_ALL.items() if f in set(plot_df["family"])]
if plot_df["representation"].map(is_residualised).any():
    handles.append(mpatches.Patch(facecolor=MUTED, hatch="//", edgecolor="white",
                                  label=f"{RESIDUALIZE_ON} projected out"))
ax.legend(handles=handles, frameon=False, loc="lower right", labelcolor=INK)
plt.tight_layout(); plt.show()

# patient_f1_cal is the leakage column: a high AUC next to a high one is not a result
lopo.join(leak_wide[[PATIENT_KEY]].rename(columns={PATIENT_KEY: "patient_f1_cal"}),
          on="representation")

#### (f) The inflammation axis, and what removing it costs

Leave-one-patient-out ridge on `Inflammation_score` at the sample level; `r_within_patient` is the honest column, and the `*_inflamresid` arms should collapse on it without losing remission AUC.

In [ ]:
# (f) leave-one-patient-out ridge on Inflammation_score, at the sample level
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr

_score_all = pd.to_numeric(meta_adata.obs[INFLAM_KEY], errors="coerce")
_pat_all   = meta_adata.obs[PATIENT_KEY].astype(str)
_ok        = _score_all.notna().to_numpy()
print(f"{int(_ok.sum())} scored samples from {_pat_all[_ok].nunique()} patients | "
      f"score {_score_all[_ok].min():.2f}-{_score_all[_ok].max():.2f}")

def lopo_regression(X, alpha=1.0):
    """Returns (pooled r, within-patient r); the second centres both on the patient mean first."""
    X = np.asarray(X, dtype=np.float64)[_ok]
    y = _score_all.to_numpy(float)[_ok]
    g = _pat_all.to_numpy()[_ok]
    pred = np.full(len(y), np.nan)
    reg = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    for tr_i, te_i in LeaveOneGroupOut().split(X, y, groups=g):
        reg.fit(X[tr_i], y[tr_i])
        pred[te_i] = reg.predict(X[te_i])

    pooled = spearmanr(y, pred).statistic
    df = pd.DataFrame({"y": y, "p": pred, "g": g})
    multi = df.groupby("g")["y"].transform("size") >= 2
    df = df[multi]
    if df["g"].nunique() < 2:
        return pooled, np.nan
    dy = df["y"] - df.groupby("g")["y"].transform("mean")
    dp = df["p"] - df.groupby("g")["p"].transform("mean")
    return pooled, spearmanr(dy, dp).statistic

rows = []
for m in bb_methods:
    r_pool, r_within = lopo_regression(meta_adata.obsm[f"{m}_representations"])
    rows.append({"representation": m, "family": rep_family(m),
                 "residualised": is_residualised(m),
                 "r_pooled": r_pool, "r_within_patient": r_within})

# references through the identical loop
for col in ["Ileum_vs_Colon", BATCH_KEY]:
    oh = pd.get_dummies(meta_adata.obs[col].astype(str)).to_numpy(dtype=float)
    r_pool, r_within = lopo_regression(oh)
    rows.append({"representation": f"reference: {col} (one-hot)", "family": "reference",
                 "residualised": False, "r_pooled": r_pool, "r_within_patient": r_within})

inflam = pd.DataFrame(rows).sort_values("r_within_patient", ascending=False).reset_index(drop=True)
REGION_REF = float(inflam.loc[inflam["representation"] == "reference: Ileum_vs_Colon (one-hot)",
                              "r_pooled"].iat[0])

print(f"\n=== LOPO ridge on {INFLAM_KEY} (sample level, split by patient) ===")
print("r_pooled is inflated by anatomy; r_within_patient is the honest column.")
display(inflam.style.format({"r_pooled": "{:+.3f}", "r_within_patient": "{:+.3f}"})
        .background_gradient(cmap="PuOr", subset=["r_pooled", "r_within_patient"],
                             vmin=-0.8, vmax=0.8))
print(f"reference -- region alone explains r_pooled = {REGION_REF:+.3f}")

_res_rows = inflam[inflam["residualised"]]
if len(_res_rows):
    print(f"\nprojection check -- {RESIDUALIZE_ON} arms should sit near 0 on both columns:")
    print(_res_rows[["representation", "r_pooled", "r_within_patient"]]
          .round(3).to_string(index=False))
    print(f"\nworst residual leakage: |r_within_patient| = "
          f"{_res_rows['r_within_patient'].abs().max():.3f}")

### SampleCLR (supervised)

`ContrastiveModel` trains a per-cell projector plus an attention aggregator, then pools each sample's cells into one vector.

In [ ]:
adata_pb

In [ ]:
# sample-level label counts
print(adata_pb.obs.drop_duplicates(SAMPLE_KEY)["Remission_status"].value_counts(dropna=False))

In [ ]:
# sample-level label counts
print(adata_pb.obs.drop_duplicates(SAMPLE_KEY)["Inflammation_binary"].value_counts(dropna=False))

In [ ]:
# sample-level label counts
print(adata_pb.obs.drop_duplicates(SAMPLE_KEY)["Ileum_vs_Colon"].value_counts(dropna=False))

In [ ]:
meta_adata.obs["Disease"].value_counts()

In [ ]:
# SampleCLR hyper-parameters
model_cfg = dict(
    num_layers=4,
    hidden_size=32,
    learning_rate_feature=3e-4,
    weight_decay=1e-5,
    n_aggregator_heads=4,
    aggregator_num_layers=2,
    aggregator_hidden_size=32,
    output_dim=64,
    classifier_num_layers=2,
    classifier_hidden_size=32,
    contrastive_loss="XSampleCLR",
    contrastive_loss_temperature=0.1,
    lambda_=1.0,
)

### Ileum colon - inflamation

In [ ]:
SEED = 0
strat = meta_adata.obs.get("Ileum_vs_Colon", meta_adata.obs["Inflammation_binary"]).astype(str).values
train_ids, val_ids = train_test_split(
    list(keep_samples), test_size=0.2, random_state=SEED, shuffle=True, stratify=strat)

In [ ]:
strat

In [ ]:
model = sampleclr.ContrastiveModel(
    adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
    train_ids=train_ids, val_ids=val_ids,
    tasks={"classification": ["Ileum_vs_Colon"]},   # <- supervised head; NaN labels auto-masked
    supervised_class_balancing="inverse",
    n_cells_per_sample=[50, 150], batch_size=32,
    early_stopping_patience=50,
    num_warmup_epochs_stage1=10, num_warmup_epochs_stage2=10,
    device=device, seed=SEED, **model_cfg,
)
model.train(num_epochs_stage1=100, num_epochs_stage2=150,
            stage1_val_metric="loss", stage2_val_metric="total", verbose=True)

In [ ]:
subset_size = int(min(counts.loc[keep_samples].max(), 512))
reps_arr = get_sample_representations_from_adata(
    projector=model.projector, aggregator=model.aggregator,
    adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
    meta_obs_names=list(meta_adata.obs_names), subset_size=subset_size,
    device=device, cell_selection="random")
store_rep("sampleclr_location", reps_arr, dist=None, samples=list(meta_adata.obs_names))

### remission - inflamation

In [ ]:
SEED = 0
strat = meta_adata.obs.get("Remission_status", meta_adata.obs["Inflammation_binary"]).astype(str).values
train_ids, val_ids = train_test_split(
    list(keep_samples), test_size=0.2, random_state=SEED, shuffle=True, stratify=strat)

In [ ]:
strat

In [ ]:
model = sampleclr.ContrastiveModel(
    adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
    train_ids=train_ids, val_ids=val_ids,
    tasks={"classification": ["Remission_status"]},   # <- supervised head; NaN labels auto-masked
    supervised_class_balancing="inverse",
    n_cells_per_sample=[50, 150], batch_size=32,
    early_stopping_patience=50,
    num_warmup_epochs_stage1=10, num_warmup_epochs_stage2=10,
    device=device, seed=SEED, **model_cfg,
)
model.train(num_epochs_stage1=100, num_epochs_stage2=150,
            stage1_val_metric="loss", stage2_val_metric="total", verbose=True)

In [ ]:
subset_size = int(min(counts.loc[keep_samples].max(), 512))
reps_arr = get_sample_representations_from_adata(
    projector=model.projector, aggregator=model.aggregator,
    adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
    meta_obs_names=list(meta_adata.obs_names), subset_size=subset_size,
    device=device, cell_selection="random")
store_rep("sampleclr_remission", reps_arr, dist=None, samples=list(meta_adata.obs_names))

### remission - site

In [ ]:
SEED = 0
strat = meta_adata.obs.get("Remission_status", meta_adata.obs["Site"]).astype(str).values
train_ids, val_ids = train_test_split(
    list(keep_samples), test_size=0.2, random_state=SEED, shuffle=True, stratify=strat)

In [ ]:
strat

In [ ]:
model = sampleclr.ContrastiveModel(
    adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
    train_ids=train_ids, val_ids=val_ids,
    tasks={"classification": ["Remission_status"]},   # <- supervised head; NaN labels auto-masked
    supervised_class_balancing="inverse",
    n_cells_per_sample=[50, 150], batch_size=32,
    early_stopping_patience=50,
    num_warmup_epochs_stage1=10, num_warmup_epochs_stage2=10,
    device=device, seed=SEED, **model_cfg,
)
model.train(num_epochs_stage1=100, num_epochs_stage2=150,
            stage1_val_metric="loss", stage2_val_metric="total", verbose=True)

In [ ]:
subset_size = int(min(counts.loc[keep_samples].max(), 512))
reps_arr = get_sample_representations_from_adata(
    projector=model.projector, aggregator=model.aggregator,
    adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
    meta_obs_names=list(meta_adata.obs_names), subset_size=subset_size,
    device=device, cell_selection="random")
store_rep("sampleclr_remission_site", reps_arr, dist=None, samples=list(meta_adata.obs_names))

### Benchmark: how well does each representation recover known covariates?

kNN over each sample-distance matrix: high on `relevant` is good, high on `technical` means the representation leaks batch.

In [ ]:
%matplotlib inline
benchmark_schema = {
    "relevant": {                        # biology we want the sample embedding to capture
        "Disease":            "classification",
        "Inflammation":       "classification",
        "Inflammation_score": "regression",
        "Treatment":          "classification",
        "Remission_status":   "classification"
    },
    "technical": {                       # nuisance; scored reversed, so higher = better
        "Batch":       "classification",
        "LibraryType": "classification",
        "n_cells":     "regression",
    },
    "contextual": {                      # fine to catch, but not the target
        "Site":             "classification",
        "Ileum_vs_Colon":   "classification",
        "Disease_duration": "regression",
        "Age":              "regression",
        "Gender":           "classification",
    },
}
methods = list(meta_adata.uns["sample_representations"])
patpy_all = pd.concat(
    [pat.tl.evaluation.knn_prediction_score(meta_adata, benchmark_schema, representations=[m])
     for m in methods],
    ignore_index=True,
)
knn_wide = patpy_all.pivot_table(index="representation", columns="covariate", values="score")
knn_wide

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=patpy_all, y="covariate", x="score", hue="representation",
            orient="h", palette="tab10", ax=ax)
ax.set_xlim(0, 1.02)
ax.set_title("kNN recovery of covariates from sample distances\n(relevant: higher = better | technical: lower = better)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout(); plt.show()

### Visualise the sample-level embedding

One UMAP over samples per representation, coloured by the sample-level covariates.

In [ ]:
sc.settings.figdir = 'hca-gut-atlas-tutorial/images/temp_images/'
sc.settings.set_figure_params(dpi=180, format='svg', transparent=True)

In [ ]:
%matplotlib inline
COLORS = [
    "Disease", "Inflammation", "Inflammation_score", "Treatment",
    "Remission_status","Batch","LibraryType","n_cells", "Site"
    ]
for rep in ["pseudobulk", "composition", "gloscope", "sampleclr_remission",
            "sclr_remission_lam3_seed0", "sclr_remission_lam10_seed0",
            "sclr_remission_batchaware_seed0",
            "sclr_inflammation_seed0",                    # the inflammation axis
            "sclr_remission_seed0_inflamresid"]:          # ...projected back out
    if f"{rep}_representations" not in meta_adata.obsm:
        continue
    ehr.pp.neighbors(meta_adata, use_rep=f"{rep}_representations", key_added=f"{rep}_nn", n_neighbors=15)
    ehr.tl.umap(meta_adata, neighbors_key=f"{rep}_nn")
    meta_adata.obsm[f"{rep}_umap"] = meta_adata.obsm["X_umap"].copy()
    ehr.pl.umap(meta_adata, color=COLORS, ncols=3, size=200, wspace=0.4,
                frameon=False,
                title=[f"{rep} · {c}" for c in COLORS], save=f"_{rep}.svg")

In [ ]:
meta_adata

In [ ]:
%matplotlib inline
# gene rankings for one (cell type, head); defaults to the strongest pair from the heatmap
_best = _long.sort_values("abs_r", ascending=False).iloc[0]
FOCUS_CT   = _best["cell_type"]
FOCUS_HEAD = _best["head"]
N_TOP_GENES = 20


print(f"focus: {FOCUS_CT}  |  {FOCUS_HEAD}  "
      f"(r={_best['r']:.2f}, FDR q={_best['q_fdr']:.3g})")

_res = correlate_and_rank(interp_dict[FOCUS_CT], corr_method="spearman",
                          n_top_genes=N_TOP_GENES, verbose=False)
plot_head_top_genes(
    _res, heads=[FOCUS_HEAD],
    figsize_per_head=(3, 4),
    save_path=f'hca-gut-atlas-tutorial/images/temp_images/{FOCUS_CT}_{FOCUS_HEAD}.svg',
    )

_rank = _res["gene_rankings"][FOCUS_HEAD]
print(f"\ntop {N_TOP_GENES} genes for {FOCUS_CT} / {FOCUS_HEAD}:")
print(_rank.head(N_TOP_GENES).round(3).to_string(index=False))

In [ ]:
%matplotlib inline
# gene rankings for one (cell type, head); defaults to the strongest pair from the heatmap
_best = _long.sort_values("abs_r", ascending=False).iloc[0]
FOCUS_CT   = "Classical Monocytes"
FOCUS_HEAD = _best["head"]
N_TOP_GENES = 20

print(f"focus: {FOCUS_CT}  |  {FOCUS_HEAD}  "
      f"(r={_best['r']:.2f}, FDR q={_best['q_fdr']:.3g})")

_res = correlate_and_rank(interp_dict[FOCUS_CT], corr_method="spearman",
                          n_top_genes=N_TOP_GENES, verbose=False)
plot_head_top_genes(_res, heads=[FOCUS_HEAD], figsize_per_head=(6, 5))

_rank = _res["gene_rankings"][FOCUS_HEAD]
print(f"\ntop {N_TOP_GENES} genes for {FOCUS_CT} / {FOCUS_HEAD}:")
print(_rank.head(N_TOP_GENES).round(3).to_string(index=False))

In [ ]:
%matplotlib inline
# sanity-check a single gene, and close the loop back to the covariate
FOCUS_GENE = _rank["gene"].iloc[0]
plot_gene_head_scatter(interp_dict[FOCUS_CT], _res, gene=FOCUS_GENE, head=FOCUS_HEAD)

# re-correlate each top gene against the covariate itself, at the same level as the heatmap
_interp = interp_dict[FOCUS_CT]
_vals_ct = _vals.reindex(_interp.sample_expression.index)
_expr = _interp.sample_expression
if GROUP_BY_PATIENT:
    _g = meta_adata.obs[PATIENT_KEY].astype(str).reindex(_expr.index)
    _expr, _vals_ct = _expr.groupby(_g).mean(), _vals_ct.groupby(_g).first()
_ok = _vals_ct.notna().to_numpy()

rows = []
for g in _rank["gene"].head(N_TOP_GENES):
    if g not in _expr.columns:
        continue
    e = _expr[g].to_numpy(float)[_ok]
    if np.unique(e).size < 2:
        continue
    r_cov, p_cov = spearmanr(_vals_ct.to_numpy(float)[_ok], e)
    rows.append({"gene": g,
                 "r_attention": float(_res["correlations"].loc[g, FOCUS_HEAD]),
                 f"r_{COVARIATE}": r_cov, "p": p_cov})
gene_cov = pd.DataFrame(rows)
gene_cov["q_fdr"] = correct_pvalues_matrix(gene_cov[["p"]], method="fdr_bh")["p"].to_numpy()

print(f"{FOCUS_CT} / {FOCUS_HEAD}: top genes re-correlated against {COVARIATE} "
      f"({'patient' if GROUP_BY_PATIENT else 'sample'}-level, n={int(_ok.sum())})")
print(gene_cov.sort_values("q_fdr").round(3).to_string(index=False))
print(f"\n{int((gene_cov['q_fdr'] < 0.05).sum())}/{len(gene_cov)} of the head's top genes are "
      f"themselves FDR<0.05 for {COVARIATE}")

# Running sampleCLR on pre treatment only

In [ ]:
adata_taurus_CD.obs['Treatment'].value_counts()

In [ ]:
adata_taurus_CD = adata_taurus_CD[adata_taurus_CD.obs['Treatment']=="Pre"]

In [ ]:
adata_taurus_CD

In [ ]:
# pipeline imports
import torch
import scipy.sparse as sp
from scipy.spatial.distance import pdist, squareform
from sklearn.model_selection import train_test_split

import patpy as pat
import ehrapy as ehr
from sampleclr.utils import get_sample_representations_from_adata

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# what to analyse: the 30-dim scANVI latent shared by reference and query
adata         = adata_taurus_CD
LAYER         = "X_scANVI_across_lin"
SAMPLE_KEY    = "sample_id"
CELLGROUP_KEY = "predictions"

assert LAYER in adata.obsm, f"{LAYER!r} not in obsm {list(adata.obsm)}"
print("device:", device, "| cells:", adata.n_obs, "| latent:", adata.obsm[LAYER].shape)

In [ ]:
adata_taurus_CD

In [ ]:
# how many samples survive each cell-count threshold
counts = adata.obs[SAMPLE_KEY].value_counts()
print("samples retained by threshold:")
for t in [20, 50, 100, 200, 300, 500]:
    print(f"  >= {t:4d} cells: {(counts >= t).sum():4d} / {len(counts)}")

In [ ]:
MIN_CELLS = 300
keep_samples = counts[counts >= MIN_CELLS].index
mask = adata.obs[SAMPLE_KEY].isin(keep_samples).to_numpy()
print(f"\nMIN_CELLS={MIN_CELLS}: {len(keep_samples)} samples, {mask.sum()} cells")

In [ ]:
# cell-level object for patpy / sampleCLR; X is empty because every method reads obsm[LAYER]
obs = adata.obs.loc[mask].copy()
emb = np.asarray(adata.obsm[LAYER][mask])
adata_pb = ad.AnnData(
    X=sp.csr_matrix((emb.shape[0], 0), dtype=np.float32),
    obs=obs, var=pd.DataFrame(index=[]), obsm={LAYER: emb},
)

In [ ]:
adata_taurus_CD.obs.columns

In [ ]:

# one row per sample: the metadata table everything attaches to
meta = (obs.drop_duplicates(subset=[SAMPLE_KEY]).set_index(SAMPLE_KEY)
        .loc[keep_samples, ['Patient', 'Disease', 'Site', 'Treatment',
       'Disease_duration', 'Inflammation','Remission_status', 'Age', 'Gender',
       'Inflammation_score','Inflammation_binary', 'Ileum_vs_Colon', 'LibraryType',
       'Batch']])
meta["n_cells"] = counts.loc[keep_samples]
meta_adata = ad.AnnData(obs=meta)
meta_adata.uns["sample_representations"] = []
print(meta_adata)
meta.head()

In [ ]:
def store_rep(name, reps, dist=None, samples=None):
    """Attach a sample representation (+cosine distances if none given) to meta_adata."""
    order = list(meta_adata.obs_names)
    if isinstance(reps, pd.DataFrame):
        reps_arr = reps.loc[order].to_numpy()
    else:
        samples = list(samples)
        pos = {s: i for i, s in enumerate(samples)}
        reps_arr = np.asarray(reps)[[pos[s] for s in order]]
    if dist is None:
        dist_df = pd.DataFrame(squareform(pdist(reps_arr, metric="cosine")), index=order, columns=order)
    elif isinstance(dist, pd.DataFrame):
        dist_df = dist.loc[order, order]
    else:
        dist_df = pd.DataFrame(np.asarray(dist), index=list(samples), columns=list(samples)).loc[order, order]
    meta_adata.obsm[f"{name}_representations"] = reps_arr
    meta_adata.obsm[f"{name}_distances"] = dist_df
    lst = meta_adata.uns.setdefault("sample_representations", [])
    if name not in lst:
        lst.append(name)
    print(f"stored {name!r}: reps {reps_arr.shape}")

### Baseline sample representations (patpy)

In [ ]:
# baseline 1: pseudobulk (mean scANVI latent per sample)
pb = pat.tl.Pseudobulk(sample_key=SAMPLE_KEY, cell_group_key=CELLGROUP_KEY, layer=LAYER)
pb.prepare_anndata(adata_pb)
pb_dist = pb.calculate_distance_matrix(force=True)

In [ ]:
store_rep("pseudobulk", pb.sample_representation, pb_dist, pb.samples)

In [ ]:
# baseline 2: cell-type composition (CLR of cell-type fractions)
comp = pat.tl.CellGroupComposition(sample_key=SAMPLE_KEY, cell_group_key=CELLGROUP_KEY, layer=LAYER)
comp.prepare_anndata(adata_pb)
comp_dist = comp.calculate_distance_matrix(force=True)

In [ ]:
store_rep("composition", comp.sample_representation, comp_dist, comp.samples)

In [ ]:
# baseline 3: GloScope (kNN density divergence); use_gpu=False if RAPIDS and torch disagree on CUDA
try:
    glo = pat.tl.GloScope_py(sample_key=SAMPLE_KEY, layer=LAYER, k=30, use_gpu=True)
    glo.prepare_anndata(adata_pb)
    glo_dist = glo.calculate_distance_matrix(force=True)
    store_rep("gloscope", glo.sample_representation, glo_dist, glo.samples)
except Exception as e:
    print("GloScope skipped:", repr(e))

In [ ]:
# baseline 4: cell-type pseudobulk (optional)
# ct_pb = pat.tl.GroupedPseudobulk(sample_key=SAMPLE_KEY, cell_group_key=CELLGROUP_KEY, layer=LAYER)
# ct_pb.prepare_anndata(adata_pb)
# ct_pb_dist = ct_pb.calculate_distance_matrix(force=True)

In [ ]:
# ct_pb.sample_representation

In [ ]:
# store_rep("CT_pseudobulk", ct_pb.sample_representation, ct_pb_dist, ct_pb.samples)

### Train SampleCLR

Remission is constant within patient (19 patients, 16 labelled) and `Batch` / `LibraryType` are confounded with it, so every split below is over patients and the batch-corrected and uncorrected numbers are read as a pair.

In [ ]:
# setup + audit: how much of remission is already explained by technical covariates
import json
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SCLR_REPO  = Path("/ictstr01/groups/ml01/workspace/christopher.lance/SampleCLR")
OUTPUT_DIR = Path("hca-gut-atlas-downstream/data/taurus_data/sampleclr_beyond_batch")
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "representations").mkdir(parents=True, exist_ok=True)

PATIENT_KEY = "Patient"
BATCH_KEY   = "Batch"
TARGET      = "Remission_clean"      # created in the next cell

# colour follows the family of a representation, never its rank
FAMILY_COLORS = {
    "baseline":       "#2a78d6",
    "zero-shot":      "#eb6834",
    "supervised":     "#1baf7a",
    "batch-corrected": "#4a3aa7",
}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

sm = meta_adata.obs
print(f"{len(sm)} samples | {sm[PATIENT_KEY].nunique()} patients | "
      f"median {sm[PATIENT_KEY].value_counts().median():.0f} samples/patient\n")

# 1. remission is a patient-level label
lab = sm["Remission_status"].astype(str).str.strip()
print(pd.DataFrame({
    "samples":  lab.value_counts(),
    "patients": sm.assign(_l=lab).groupby("_l", observed=True)[PATIENT_KEY].nunique(),
}).to_string(), "\n")

n_multi = sm.assign(_l=lab).groupby(PATIENT_KEY, observed=True)["_l"].nunique().gt(1).sum()
print(f"patients with more than one remission value: {int(n_multi)}"
      "  <- 0 means the label carries no within-patient information\n")

# 2. how much of the outcome is already in the technical covariates?
def majority_rule_accuracy(feature, target):
    """Accuracy of 'predict the majority target class within each level of feature'."""
    df = pd.DataFrame({"f": feature.astype(str), "t": target.astype(str)}).dropna()
    pred = df.groupby("f", observed=True)["t"].transform(lambda s: s.mode().iat[0])
    return (pred == df["t"]).mean()

cd = lab.isin(["Remission", "Non_Remission"]).to_numpy()      # drop healthy
print(f"majority-class baseline                 : {lab[cd].value_counts(normalize=True).max():.3f}")
for c in [BATCH_KEY, "LibraryType", "Site", "Ileum_vs_Colon", PATIENT_KEY]:
    print(f"predict remission from {c:<17}: {majority_rule_accuracy(sm.loc[cd, c], lab[cd]):.3f}")

print("\n--- Batch x Remission (samples) ---")
print(pd.crosstab(sm.loc[cd, BATCH_KEY].astype(str), lab[cd]).to_string())
print("\n--- LibraryType x Remission (samples) ---")
print(pd.crosstab(sm.loc[cd, "LibraryType"].astype(str), lab[cd]).to_string())
print("\n--- patients per Batch (Batch is nested inside Patient) ---")
print(sm.groupby(BATCH_KEY, observed=True)[PATIENT_KEY].nunique().to_string())

#### (a) Clean the target

The "no label" level is the literal string `'None '`; mapping it to `NaN` keeps healthy samples in the contrastive loss but out of the classification head.

In [ ]:
# (a) clean the target: 'None ' / 'Not_avail' -> NaN
REMISSION_NA = {"None", "Not_avail", "nan", "NaN", "<NA>", ""}

def clean_remission(s):
    s = s.astype(str).str.strip()
    return s.where(~s.isin(REMISSION_NA), np.nan)

for _ad in (adata_pb, meta_adata):
    _ad.obs[TARGET] = clean_remission(_ad.obs["Remission_status"])
    # drop any stale encoding so SampleCLR re-encodes from the cleaned column
    _ad.obs.drop(columns=[f"{TARGET}_encoded"], errors="ignore", inplace=True)

print("cells   :", adata_pb.obs[TARGET].value_counts(dropna=False).to_dict())
print("samples :", meta_adata.obs[TARGET].value_counts(dropna=False).to_dict())
print("labelled patients:",
      meta_adata.obs.loc[meta_adata.obs[TARGET].notna(), PATIENT_KEY].nunique(),
      "of", meta_adata.obs[PATIENT_KEY].nunique())

#### (b) Patient-grouped train / val / test

Splits are drawn over patients and expanded to sample IDs; `test_ids` is a held-back reserve, the real scoring is the leave-one-patient-out pass below.

In [ ]:
# (b) patient-grouped splits, expanded to sample IDs
def patient_split(seed=0, test_frac=0.25, val_frac=0.25, verbose=True):
    """Split patients (not samples) into train/val/test; returns three lists of sample IDs."""
    obs = meta_adata.obs
    lab_by_pat = (obs.loc[obs[TARGET].notna()]
                     .groupby(PATIENT_KEY, observed=True)[TARGET].first())
    pats, y = lab_by_pat.index.to_numpy(), lab_by_pat.to_numpy()

    rest_i, test_i = next(StratifiedShuffleSplit(
        n_splits=1, test_size=test_frac, random_state=seed).split(pats, y))
    tr_i, val_i = next(StratifiedShuffleSplit(
        n_splits=1, test_size=val_frac, random_state=seed).split(pats[rest_i], y[rest_i]))

    test_pats  = set(pats[test_i])
    val_pats   = set(pats[rest_i][val_i])
    train_pats = (set(pats[rest_i][tr_i])
                  | set(obs.loc[obs[TARGET].isna(), PATIENT_KEY].astype(str).unique()))

    # a patient must live in exactly one split -- this is the leakage guard
    assert not (train_pats & val_pats), train_pats & val_pats
    assert not (train_pats & test_pats), train_pats & test_pats
    assert not (val_pats & test_pats),   val_pats & test_pats

    def ids(p):
        return obs.index[obs[PATIENT_KEY].astype(str).isin({str(x) for x in p})].tolist()

    splits = {"train": (train_pats, ids(train_pats)),
              "val":   (val_pats,   ids(val_pats)),
              "test":  (test_pats,  ids(test_pats))}
    if verbose:
        for name, (p, s) in splits.items():
            print(f"  {name:5s}: {len(p):2d} patients, {len(s):3d} samples  "
                  f"{obs.loc[s, TARGET].value_counts().to_dict()}")
    return [splits[k][1] for k in ("train", "val", "test")]

train_ids, val_ids, test_ids = patient_split(seed=0)
assert not (set(train_ids) & set(val_ids) & set(test_ids))
print(f"\n{len(train_ids) + len(val_ids) + len(test_ids)} of {meta_adata.n_obs} samples assigned")

#### (c) Model config + ablation grid

Grid over supervision (none / remission / inflammation), batch-aware sampling and `lambda_`, plus derived arms with `Inflammation_score` projected out post-hoc.

In [ ]:
# (c) model config + ablation grid
SCLR_SIZE = "tiny"                       # "tiny" | "medium" | "large"
sclr_cfg  = json.loads((SCLR_REPO / f"configs/models/{SCLR_SIZE}.json").read_text())
sclr_cfg["contrastive_loss_temperature"] = 0.1
sclr_cfg.pop("batch_size", None)                      # passed explicitly below
print(f"{SCLR_SIZE}.json ->", json.dumps(sclr_cfg, indent=2))

CELL_SELECTION   = "random"              # 'eigenvector' needs cugraph
BASE_TASKS       = {"classification": [TARGET]}
INFLAM_KEY       = "Inflammation_score"  # continuous histology score, one value per sample
INFLAM_TASKS     = {"regression": [INFLAM_KEY]}
LAMBDA_DEFAULT   = 1.0
RUN_LAMBDA_SWEEP = True
LAMBDA_SWEEP     = [0.5, 1.0, 3.0, 10.0]
N_SEEDS          = 1                     # raise to average over split draws

CONFIGS = [
    dict(rep_name="sclr_zeroshot",             tasks=None,         batch_aware=False, lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_zeroshot_batchaware",  tasks=None,         batch_aware=True,  lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_remission",            tasks=BASE_TASKS,   batch_aware=False, lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_remission_batchaware", tasks=BASE_TASKS,   batch_aware=True,  lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_remission_batchcorr",  batch_aware=False,  lambda_=LAMBDA_DEFAULT,
         tasks={**BASE_TASKS, "batch_correction": BATCH_KEY}),
    # inflammation severity alone: a biopsy-level, within-patient axis, orthogonal to remission
    dict(rep_name="sclr_inflammation",         tasks=INFLAM_TASKS, batch_aware=False, lambda_=LAMBDA_DEFAULT),
]
if RUN_LAMBDA_SWEEP:
    CONFIGS += [dict(rep_name=f"sclr_remission_lam{l:g}".replace(".", "p"),
                     tasks=BASE_TASKS, batch_aware=False, lambda_=l)
                for l in LAMBDA_SWEEP if l != LAMBDA_DEFAULT]

# derived arms: Inflammation_score projected out post-hoc, baselines included as the reference
RESIDUALIZE_ON = INFLAM_KEY
RESIDUALIZE    = ["pseudobulk", "composition",
                  "sclr_remission", "sclr_remission_batchaware", "sclr_remission_batchcorr"]
RESID_SUFFIX   = "_inflamresid"

# residualisation is a secondary encoding (marker shape / hatch); an arm keeps its parent's colour
def is_residualised(name):
    return RESID_SUFFIX in name

def rep_family(name):
    n = name.lower().replace(RESID_SUFFIX, "")          # residualised keeps its parent's family
    if not n.startswith("sclr"):                        return "baseline"
    if "batchcorr" in n or "batchaware" in n:           return "batch-corrected"
    if "zeroshot" in n:                                 return "zero-shot"
    return "supervised"

print(f"\n{len(CONFIGS)} configs x {N_SEEDS} seed(s) = {len(CONFIGS) * N_SEEDS} models")
for c in CONFIGS:
    print(f"  {c['rep_name']:34s} tasks={str(c['tasks']):55s} "
          f"batch_aware={c['batch_aware']!s:5s} lambda={c['lambda_']}")
print(f"\n+ {len(RESIDUALIZE)} derived arms (no training): {RESIDUALIZE_ON} projected out of")
for b in RESIDUALIZE:
    print(f"  {b}{RESID_SUFFIX}")

In [ ]:
# train the grid, skipping anything already cached on disk
import contextlib, io, hashlib

subset_size = int(min(counts.loc[keep_samples].max(), 512))
bb_reps = []
splits_by_seed = {}          # seed -> train sample IDs, so residualisation uses the right split

def cfg_fingerprint(cfg, seed):
    """Hash of everything that changes the model, so a stale cached representation is caught."""
    payload = json.dumps({"cfg": cfg, "seed": seed, "sclr_cfg": sclr_cfg,
                          "layer": LAYER, "min_cells": MIN_CELLS},
                         sort_keys=True, default=str)
    return hashlib.sha1(payload.encode()).hexdigest()[:12]

for seed in range(N_SEEDS):
    tr, va, te = patient_split(seed=seed, verbose=(seed == 0))
    splits_by_seed[seed] = tr

    for cfg in CONFIGS:
        rep_name  = f"{cfg['rep_name']}_seed{seed}"
        rep_path  = OUTPUT_DIR / "representations" / f"{rep_name}_representations.npy"
        ckpt_path = OUTPUT_DIR / "models" / f"{rep_name}.pt"
        fingerprint = cfg_fingerprint(cfg, seed)

        if rep_path.exists():
            cached_fp = (torch.load(ckpt_path, map_location="cpu", weights_only=False).get("fingerprint")
                         if ckpt_path.exists() else None)
            if cached_fp is not None and cached_fp != fingerprint:
                print(f"!! {rep_name}: cached representation was trained under a different config "
                      f"({cached_fp} != {fingerprint}). Delete {rep_path.name} to retrain; "
                      f"loading the stale one for now.")
            store_rep(rep_name, np.load(rep_path), dist=None, samples=list(meta_adata.obs_names))
            bb_reps.append(rep_name)
            continue

        print(f"\n{'=' * 78}\n{rep_name}  |  tasks={cfg['tasks']}  "
              f"batch_aware={cfg['batch_aware']}  lambda_={cfg['lambda_']}\n{'=' * 78}")

        kw = dict(
            adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
            train_ids=tr, val_ids=va, test_ids=te,
            supervised_class_balancing="inverse",     # 68 Remission vs 28 Non_Remission samples
            lambda_=cfg["lambda_"],
            n_cells_per_sample=[50, 150], batch_size=32,
            early_stopping_patience=50,
            num_warmup_epochs_stage1=10, num_warmup_epochs_stage2=10,
            device=device, seed=seed, verbose=False, **sclr_cfg,
        )
        if cfg["tasks"] is not None:
            kw["tasks"] = cfg["tasks"]
        if cfg["batch_aware"]:
            kw.update(use_batch_aware_sampler=True,
                      batch_sampler_batch_col=BATCH_KEY,
                      # Batch_4 has only 5 samples -- merge undersized batches for the sampler
                      batch_sampler_pseudo_batch_strategy="similarity")

        model = sampleclr.ContrastiveModel(**kw)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            model.train(num_epochs_stage1=100, num_epochs_stage2=150,
                        stage1_val_metric="loss",
                        stage2_val_metric="total" if cfg["tasks"] else "loss",
                        verbose=False)

        reps_arr = get_sample_representations_from_adata(
            projector=model.projector, aggregator=model.aggregator,
            adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
            meta_obs_names=list(meta_adata.obs_names), subset_size=subset_size,
            device=device, cell_selection=CELL_SELECTION)

        store_rep(rep_name, reps_arr, dist=None, samples=list(meta_adata.obs_names))
        bb_reps.append(rep_name)

        np.save(rep_path, reps_arr)
        torch.save({
            "projector":  model.projector.state_dict(),
            "aggregator": model.aggregator.state_dict(),
            # every head group is saved, so load_sclr_model can restore any arm
            "classifiers":        [c.state_dict() for c in getattr(model, "classifiers", [])],
            "regressors":         [r.state_dict() for r in getattr(model, "regressors", [])],
            "ordinal_regressors": [o.state_dict() for o in getattr(model, "ordinal_regressors", [])],
            "cfg": cfg, "seed": seed, "layer": model.layer, "output_dim": model.output_dim,
            "sclr_cfg": sclr_cfg, "fingerprint": fingerprint,
            "split": {"train": tr, "val": va, "test": te},
        }, ckpt_path)

meta_adata.obs["split"] = "train"
meta_adata.obs.loc[val_ids,  "split"] = "val"
meta_adata.obs.loc[test_ids, "split"] = "test"
print("\ntrained/loaded:", bb_reps)

# derived arms: project Inflammation_score out of an existing representation
def residualise(X, values, fit_mask):
    """OLS removal of the linear `values` component, with coefficients fit on train patients only."""
    X = np.asarray(X, dtype=np.float64)
    v = np.asarray(values, dtype=np.float64)
    D = np.column_stack([np.ones(len(v)), v])
    ok = fit_mask & np.isfinite(v) & np.isfinite(X).all(axis=1)
    coef, *_ = np.linalg.lstsq(D[ok], X[ok], rcond=None)
    return (X - D @ coef).astype(np.float32)

_score = pd.to_numeric(meta_adata.obs[RESIDUALIZE_ON], errors="coerce").to_numpy(float)
assert np.isfinite(_score).all(), \
    f"{RESIDUALIZE_ON} has {int(np.isnan(_score).sum())} NaN; the projection would be undefined there"
print(f"\nprojecting out {RESIDUALIZE_ON} (coefficients fit on train patients, applied to all "
      f"{len(_score)} samples)")

resid_reps = []
for base in RESIDUALIZE:
    # a seedless name means a patpy baseline (one copy); an sclr arm has one per seed
    names = ([base] if f"{base}_representations" in meta_adata.obsm
             else [f"{base}_seed{s}" for s in range(N_SEEDS)])
    for name in names:
        if f"{name}_representations" not in meta_adata.obsm:
            print(f"  skip {name}: no representation stored")
            continue
        # baselines carry no seed, so they are residualised on the seed-0 split
        seed = int(name.rsplit("_seed", 1)[1]) if "_seed" in name else 0
        fit  = meta_adata.obs_names.isin(splits_by_seed.get(seed, train_ids))
        out  = f"{name}{RESID_SUFFIX}"
        store_rep(out, residualise(meta_adata.obsm[f"{name}_representations"], _score, fit),
                  dist=None, samples=list(meta_adata.obs_names))
        resid_reps.append(out)

bb_reps += resid_reps
print("\nderived:", resid_reps)

#### (d) Benchmark, with `Patient` as the leakage detector

kNN recovery per covariate (`f1_macro_calibrated`: 0 = chance, 1 = perfect), with `Patient` scored uninverted so remission recovery that is only patient re-identification stays visible.

In [ ]:
# (d) benchmark: covariate recovery, with Patient as the leakage detector
bb_baselines = [m for m in ["pseudobulk", "composition", "gloscope", "CT_pseudobulk"]
                if f"{m}_representations" in meta_adata.obsm]
bb_methods   = bb_baselines + bb_reps
print("scoring:", bb_methods)

benchmark_schema_bb = {
    "relevant": {
        TARGET:         "classification",
        "Treatment":    "classification",
        "Inflammation": "classification",
        INFLAM_KEY:     "regression",
    },
    "technical": {                          # reported inverted: higher = less leakage
        PATIENT_KEY:   "classification",
        BATCH_KEY:     "classification",
        "LibraryType": "classification",
        "n_cells":     "regression",
    },
    "contextual": {
        "Ileum_vs_Colon": "classification",
        "Site":           "classification",
    },
}

# same covariates under "relevant" -> RAW recovery, which is what a diagnostic needs
leak_schema = {"relevant": {PATIENT_KEY:   "classification",
                            BATCH_KEY:     "classification",
                            "LibraryType": "classification",
                            TARGET:        "classification",
                            INFLAM_KEY:    "regression"}}

bb_scores = pd.concat(
    [pat.tl.evaluation.knn_prediction_score(meta_adata, benchmark_schema_bb, representations=[m])
     for m in bb_methods], ignore_index=True)
bb_leak = pd.concat(
    [pat.tl.evaluation.knn_prediction_score(meta_adata, leak_schema, representations=[m])
     for m in bb_methods], ignore_index=True)

bb_wide   = bb_scores.pivot_table(index="representation", columns="covariate", values="score")
leak_wide = bb_leak.pivot_table(index="representation", columns="covariate", values="score")
leak_wide = leak_wide.loc[bb_methods]        # keep grid order, not alphabetical

print("\n=== uninverted kNN recovery, f1_macro_calibrated: 0 = chance, 1 = perfect ===")
display(leak_wide[[TARGET, PATIENT_KEY, BATCH_KEY, "LibraryType"]]
        .rename(columns={TARGET: "Remission"})
        .style.format("{:.3f}").background_gradient(cmap="PiYG", vmin=0, vmax=1))

# did the projection remove Inflammation_score, and what did it cost? (regression -> signed spearman r)
_pairs = [(b, f"{b}{RESID_SUFFIX}") for b in leak_wide.index if f"{b}{RESID_SUFFIX}" in leak_wide.index]
if _pairs:
    removal = pd.DataFrame(
        [{"representation":        base,
          f"{INFLAM_KEY} (before)": leak_wide.loc[base, INFLAM_KEY],
          f"{INFLAM_KEY} (after)":  leak_wide.loc[res,  INFLAM_KEY],
          "Remission (before)":     leak_wide.loc[base, TARGET],
          "Remission (after)":      leak_wide.loc[res,  TARGET],
          PATIENT_KEY:              leak_wide.loc[res,  PATIENT_KEY]}
         for base, res in _pairs]).set_index("representation")
    removal["Remission cost"] = removal["Remission (after)"] - removal["Remission (before)"]

    print(f"\n=== {RESIDUALIZE_ON} removal check ===")
    print("left pair : spearman_r. Should collapse in magnitude. A small negative value is")
    print("            expected, not a bug -- the projection is fit on train patients only,")
    print("            so held-out samples end up slightly over-corrected.")
    print("right pair: f1_macro_calibrated, the price paid in remission recovery.")
    display(removal.style.format("{:+.3f}")
            .background_gradient(cmap="PuOr", subset=[f"{INFLAM_KEY} (before)",
                                                      f"{INFLAM_KEY} (after)"], vmin=-1, vmax=1)
            .background_gradient(cmap="PiYG", subset=["Remission cost"], vmin=-0.5, vmax=0.5))
else:
    print(f"\nno {RESID_SUFFIX} arms stored -- run the training cell first")

In [ ]:
%matplotlib inline
# does the Remission score track patient leakage? signal lives above the diagonal
fig, ax = plt.subplots(figsize=(7.2, 6.4))
ax.set_facecolor("none")

lim = max(0.35, leak_wide[[TARGET, PATIENT_KEY]].to_numpy().max() * 1.25)
ax.plot([0, lim], [0, lim], color=GRID, lw=2, zorder=1)
ax.annotate("remission = patient identity", xy=(lim * 0.62, lim * 0.62),
            xytext=(6, -14), textcoords="offset points", color=MUTED, fontsize=9)

# connector from each base arm to its residualised twin
for base in leak_wide.index:
    res = f"{base}{RESID_SUFFIX}"
    if res in leak_wide.index:
        ax.plot(leak_wide.loc[[base, res], PATIENT_KEY], leak_wide.loc[[base, res], TARGET],
                color=FAMILY_COLORS[rep_family(base)], lw=1.2, alpha=0.55, zorder=2)

seen = set()
for rep in leak_wide.index:
    fam  = rep_family(rep)
    resid = is_residualised(rep)
    x, y = leak_wide.loc[rep, PATIENT_KEY], leak_wide.loc[rep, TARGET]
    ax.scatter(x, y, s=110, color=FAMILY_COLORS[fam], marker="D" if resid else "o",
               edgecolor="white", linewidth=2, zorder=3, label=fam if fam not in seen else None)
    seen.add(fam)
    ax.annotate(rep.replace("_seed0", "").replace("sclr_", "").replace(RESID_SUFFIX, " -infl"),
                xy=(x, y), xytext=(8, -3), textcoords="offset points", fontsize=9, color=INK)

ax.set_xlabel("Patient recovery  (f1_macro_calibrated, 0 = chance)", color=INK)
ax.set_ylabel("Remission recovery  (f1_macro_calibrated, 0 = chance)", color=INK)
ax.set_title("Remission signal vs patient leakage\nabove the diagonal = beyond patient identity",
             color=INK, loc="left")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.grid(True, color=GRID, lw=0.8, zorder=0)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color("#c3c2b7")
ax.tick_params(colors=MUTED)

# two legends: colour = family, shape = whether Inflammation_score was projected out
leg_fam = ax.legend(frameon=False, loc="upper right", labelcolor=INK)
ax.add_artist(leg_fam)
shape_handles = [plt.Line2D([], [], marker=m, ls="", ms=9, color=MUTED, mec="white", mew=1.5, label=l)
                 for m, l in [("o", "as trained"), ("D", f"{RESIDUALIZE_ON} projected out")]]
ax.legend(handles=shape_handles, frameon=False, loc="lower right", labelcolor=INK, fontsize=9)
plt.tight_layout(); plt.show()

#### (e) Patient-level leave-one-patient-out evaluation

Sample embeddings pooled per patient, then leave-one-patient-out logistic regression; one-hot `Batch` goes through the same loop as the reference to beat.

In [ ]:
# (e) leave-one-patient-out at the patient level
_lab_ok = meta_adata.obs[TARGET].notna().to_numpy()
_pat    = meta_adata.obs[PATIENT_KEY].astype(str).to_numpy()[_lab_ok]
_y      = (meta_adata.obs[TARGET].to_numpy()[_lab_ok] == "Remission").astype(int)

y_pat = pd.Series(_y, index=_pat).groupby(level=0).first()
print(f"{len(y_pat)} labelled patients: "
      f"{int(y_pat.sum())} Remission / {int((1 - y_pat).sum())} Non_Remission")

def to_patient_matrix(X):
    """Mean-pool sample embeddings within each patient; rows aligned to y_pat."""
    return pd.DataFrame(np.asarray(X)[_lab_ok], index=_pat).groupby(level=0).mean().loc[y_pat.index]

def lopo_scores(Xp, yp, C=1.0):
    """Leave-one-patient-out CV; returns (auc, balanced_accuracy) over pooled predictions."""
    prob = np.full(len(Xp), np.nan)
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    for tr, te in LeaveOneGroupOut().split(Xp, yp, groups=np.asarray(Xp.index)):
        clf.fit(Xp.iloc[tr], yp.iloc[tr])
        prob[te] = clf.predict_proba(Xp.iloc[te])[:, 1]
    return roc_auc_score(yp, prob), balanced_accuracy_score(yp, (prob >= 0.5).astype(int))

rows = []
for m in bb_methods:
    auc, bacc = lopo_scores(to_patient_matrix(meta_adata.obsm[f"{m}_representations"]), y_pat)
    rows.append({"representation": m, "family": rep_family(m), "AUC": auc, "balanced_acc": bacc})

# technical-covariate references, through the identical LOPO loop
for col in [BATCH_KEY, "LibraryType"]:
    oh = pd.get_dummies(meta_adata.obs[col].astype(str)).to_numpy(dtype=float)
    auc, bacc = lopo_scores(to_patient_matrix(oh), y_pat)
    rows.append({"representation": f"reference: {col} (one-hot)", "family": "reference",
                 "AUC": auc, "balanced_acc": bacc})
rows.append({"representation": "reference: chance", "family": "reference",
             "AUC": 0.5, "balanced_acc": 0.5})

lopo = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
BATCH_REF = float(lopo.loc[lopo["representation"] == f"reference: {BATCH_KEY} (one-hot)", "AUC"].iat[0])
display(lopo.style.format({"AUC": "{:.3f}", "balanced_acc": "{:.3f}"})
        .background_gradient(cmap="PiYG", subset=["AUC"], vmin=0.3, vmax=1.0))
print(f"\nreference to beat -- {BATCH_KEY} alone: AUC {BATCH_REF:.3f}")
print("clears it:", lopo.loc[(lopo.family != "reference") & (lopo.AUC > BATCH_REF),
                             "representation"].tolist() or "none")

In [ ]:
%matplotlib inline
import matplotlib.patches as mpatches
# patient-level LOPO AUC against the technical references (hatched = Inflammation_score removed)
FAM_COLORS_ALL = {**FAMILY_COLORS, "reference": MUTED}

plot_df = lopo.sort_values("AUC")            # ascending -> best on top of an hbar
fig, ax = plt.subplots(figsize=(8.4, 0.42 * len(plot_df) + 1.8))
ax.set_facecolor("none")

bars = ax.barh(plot_df["representation"].str.replace("_seed0", "", regex=False),
               plot_df["AUC"],
               color=[FAM_COLORS_ALL[f] for f in plot_df["family"]],
               hatch=["//" if is_residualised(r) else "" for r in plot_df["representation"]],
               edgecolor="white", linewidth=0,
               height=0.62, zorder=3)

ax.set_ylim(-0.6, len(plot_df) - 0.5 + 0.95)
ax.axvline(0.5, color=GRID, lw=2, zorder=1)
ax.axvline(BATCH_REF, color=MUTED, lw=2, ls=(0, (4, 3)), zorder=2)
ax.annotate(f"{BATCH_KEY} alone ({BATCH_REF:.2f})", xy=(BATCH_REF, len(plot_df) - 0.1),
            xytext=(5, 0), textcoords="offset points",
            color=MUTED, fontsize=9, va="center")

for b, v in zip(bars, plot_df["AUC"]):
    ax.text(v + 0.008, b.get_y() + b.get_height() / 2, f"{v:.2f}",
            va="center", fontsize=9, color=INK)

ax.set_xlim(0, 1.06)
ax.set_xlabel("Leave-one-patient-out AUC  (16 patients)", color=INK)
ax.set_title("Remission prediction from patient-pooled sample embeddings",
             color=INK, loc="left")
ax.grid(True, axis="x", color=GRID, lw=0.8, zorder=0)
ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(colors=MUTED)
ax.tick_params(axis="y", length=0, labelcolor=INK)

handles = [plt.Line2D([], [], marker="s", ls="", ms=9, color=c, label=f)
           for f, c in FAM_COLORS_ALL.items() if f in set(plot_df["family"])]
if plot_df["representation"].map(is_residualised).any():
    handles.append(mpatches.Patch(facecolor=MUTED, hatch="//", edgecolor="white",
                                  label=f"{RESIDUALIZE_ON} projected out"))
ax.legend(handles=handles, frameon=False, loc="lower right", labelcolor=INK)
plt.tight_layout(); plt.show()

# patient_f1_cal is the leakage column: a high AUC next to a high one is not a result
lopo.join(leak_wide[[PATIENT_KEY]].rename(columns={PATIENT_KEY: "patient_f1_cal"}),
          on="representation")

#### (f) The inflammation axis, and what removing it costs

Leave-one-patient-out ridge on `Inflammation_score` at the sample level; `r_within_patient` is the honest column, and the `*_inflamresid` arms should collapse on it without losing remission AUC.

In [ ]:
# (f) leave-one-patient-out ridge on Inflammation_score, at the sample level
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr

_score_all = pd.to_numeric(meta_adata.obs[INFLAM_KEY], errors="coerce")
_pat_all   = meta_adata.obs[PATIENT_KEY].astype(str)
_ok        = _score_all.notna().to_numpy()
print(f"{int(_ok.sum())} scored samples from {_pat_all[_ok].nunique()} patients | "
      f"score {_score_all[_ok].min():.2f}-{_score_all[_ok].max():.2f}")

def lopo_regression(X, alpha=1.0):
    """Returns (pooled r, within-patient r); the second centres both on the patient mean first."""
    X = np.asarray(X, dtype=np.float64)[_ok]
    y = _score_all.to_numpy(float)[_ok]
    g = _pat_all.to_numpy()[_ok]
    pred = np.full(len(y), np.nan)
    reg = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    for tr_i, te_i in LeaveOneGroupOut().split(X, y, groups=g):
        reg.fit(X[tr_i], y[tr_i])
        pred[te_i] = reg.predict(X[te_i])

    pooled = spearmanr(y, pred).statistic
    df = pd.DataFrame({"y": y, "p": pred, "g": g})
    multi = df.groupby("g")["y"].transform("size") >= 2
    df = df[multi]
    if df["g"].nunique() < 2:
        return pooled, np.nan
    dy = df["y"] - df.groupby("g")["y"].transform("mean")
    dp = df["p"] - df.groupby("g")["p"].transform("mean")
    return pooled, spearmanr(dy, dp).statistic

rows = []
for m in bb_methods:
    r_pool, r_within = lopo_regression(meta_adata.obsm[f"{m}_representations"])
    rows.append({"representation": m, "family": rep_family(m),
                 "residualised": is_residualised(m),
                 "r_pooled": r_pool, "r_within_patient": r_within})

# references through the identical loop
for col in ["Ileum_vs_Colon", BATCH_KEY]:
    oh = pd.get_dummies(meta_adata.obs[col].astype(str)).to_numpy(dtype=float)
    r_pool, r_within = lopo_regression(oh)
    rows.append({"representation": f"reference: {col} (one-hot)", "family": "reference",
                 "residualised": False, "r_pooled": r_pool, "r_within_patient": r_within})

inflam = pd.DataFrame(rows).sort_values("r_within_patient", ascending=False).reset_index(drop=True)
REGION_REF = float(inflam.loc[inflam["representation"] == "reference: Ileum_vs_Colon (one-hot)",
                              "r_pooled"].iat[0])

print(f"\n=== LOPO ridge on {INFLAM_KEY} (sample level, split by patient) ===")
print("r_pooled is inflated by anatomy; r_within_patient is the honest column.")
display(inflam.style.format({"r_pooled": "{:+.3f}", "r_within_patient": "{:+.3f}"})
        .background_gradient(cmap="PuOr", subset=["r_pooled", "r_within_patient"],
                             vmin=-0.8, vmax=0.8))
print(f"reference -- region alone explains r_pooled = {REGION_REF:+.3f}")

_res_rows = inflam[inflam["residualised"]]
if len(_res_rows):
    print(f"\nprojection check -- {RESIDUALIZE_ON} arms should sit near 0 on both columns:")
    print(_res_rows[["representation", "r_pooled", "r_within_patient"]]
          .round(3).to_string(index=False))
    print(f"\nworst residual leakage: |r_within_patient| = "
          f"{_res_rows['r_within_patient'].abs().max():.3f}")

## SampleCLR without inflammation removal

In [ ]:
# pipeline imports
import torch
import scipy.sparse as sp
from scipy.spatial.distance import pdist, squareform
from sklearn.model_selection import train_test_split

import patpy as pat
import ehrapy as ehr
from sampleclr.utils import get_sample_representations_from_adata

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# what to analyse: the 30-dim scANVI latent shared by reference and query
adata         = adata_taurus_CD
LAYER         = "X_scANVI_across_lin"
SAMPLE_KEY    = "sample_id"
CELLGROUP_KEY = "predictions"

assert LAYER in adata.obsm, f"{LAYER!r} not in obsm {list(adata.obsm)}"
print("device:", device, "| cells:", adata.n_obs, "| latent:", adata.obsm[LAYER].shape)

In [ ]:
adata_taurus_CD

In [ ]:
# how many samples survive each cell-count threshold
counts = adata.obs[SAMPLE_KEY].value_counts()
print("samples retained by threshold:")
for t in [20, 50, 100, 200, 300, 500]:
    print(f"  >= {t:4d} cells: {(counts >= t).sum():4d} / {len(counts)}")

In [ ]:
MIN_CELLS = 300
keep_samples = counts[counts >= MIN_CELLS].index
mask = adata.obs[SAMPLE_KEY].isin(keep_samples).to_numpy()
print(f"\nMIN_CELLS={MIN_CELLS}: {len(keep_samples)} samples, {mask.sum()} cells")

In [ ]:
# cell-level object for patpy / sampleCLR; X is empty because every method reads obsm[LAYER]
obs = adata.obs.loc[mask].copy()
emb = np.asarray(adata.obsm[LAYER][mask])
adata_pb = ad.AnnData(
    X=sp.csr_matrix((emb.shape[0], 0), dtype=np.float32),
    obs=obs, var=pd.DataFrame(index=[]), obsm={LAYER: emb},
)

In [ ]:
adata_taurus_CD.obs.columns

In [ ]:

# one row per sample: the metadata table everything attaches to
meta = (obs.drop_duplicates(subset=[SAMPLE_KEY]).set_index(SAMPLE_KEY)
        .loc[keep_samples, ['Patient', 'Disease', 'Site', 'Treatment',
       'Disease_duration', 'Inflammation','Remission_status', 'Age', 'Gender',
       'Inflammation_score','Inflammation_binary', 'Ileum_vs_Colon', 'LibraryType',
       'Batch']])
meta["n_cells"] = counts.loc[keep_samples]
meta_adata = ad.AnnData(obs=meta)
meta_adata.uns["sample_representations"] = []
print(meta_adata)
meta.head()

In [ ]:
def store_rep(name, reps, dist=None, samples=None):
    """Attach a sample representation (+cosine distances if none given) to meta_adata."""
    order = list(meta_adata.obs_names)
    if isinstance(reps, pd.DataFrame):
        reps_arr = reps.loc[order].to_numpy()
    else:
        samples = list(samples)
        pos = {s: i for i, s in enumerate(samples)}
        reps_arr = np.asarray(reps)[[pos[s] for s in order]]
    if dist is None:
        dist_df = pd.DataFrame(squareform(pdist(reps_arr, metric="cosine")), index=order, columns=order)
    elif isinstance(dist, pd.DataFrame):
        dist_df = dist.loc[order, order]
    else:
        dist_df = pd.DataFrame(np.asarray(dist), index=list(samples), columns=list(samples)).loc[order, order]
    meta_adata.obsm[f"{name}_representations"] = reps_arr
    meta_adata.obsm[f"{name}_distances"] = dist_df
    lst = meta_adata.uns.setdefault("sample_representations", [])
    if name not in lst:
        lst.append(name)
    print(f"stored {name!r}: reps {reps_arr.shape}")

### Baseline sample representations (patpy)

In [ ]:
# baseline 1: pseudobulk (mean scANVI latent per sample)
pb = pat.tl.Pseudobulk(sample_key=SAMPLE_KEY, cell_group_key=CELLGROUP_KEY, layer=LAYER)
pb.prepare_anndata(adata_pb)
pb_dist = pb.calculate_distance_matrix(force=True)

In [ ]:
store_rep("pseudobulk", pb.sample_representation, pb_dist, pb.samples)

In [ ]:
# baseline 2: cell-type composition (CLR of cell-type fractions)
comp = pat.tl.CellGroupComposition(sample_key=SAMPLE_KEY, cell_group_key=CELLGROUP_KEY, layer=LAYER)
comp.prepare_anndata(adata_pb)
comp_dist = comp.calculate_distance_matrix(force=True)

In [ ]:
store_rep("composition", comp.sample_representation, comp_dist, comp.samples)

In [ ]:
# baseline 3: GloScope (kNN density divergence); use_gpu=False if RAPIDS and torch disagree on CUDA
try:
    glo = pat.tl.GloScope_py(sample_key=SAMPLE_KEY, layer=LAYER, k=30, use_gpu=True)
    glo.prepare_anndata(adata_pb)
    glo_dist = glo.calculate_distance_matrix(force=True)
    store_rep("gloscope", glo.sample_representation, glo_dist, glo.samples)
except Exception as e:
    print("GloScope skipped:", repr(e))

In [ ]:
# baseline 4: cell-type pseudobulk (optional)
# ct_pb = pat.tl.GroupedPseudobulk(sample_key=SAMPLE_KEY, cell_group_key=CELLGROUP_KEY, layer=LAYER)
# ct_pb.prepare_anndata(adata_pb)
# ct_pb_dist = ct_pb.calculate_distance_matrix(force=True)

In [ ]:
# ct_pb.sample_representation

In [ ]:
# store_rep("CT_pseudobulk", ct_pb.sample_representation, ct_pb_dist, ct_pb.samples)

### Train SamplCLR

Remission is constant within patient (19 patients, 16 labelled) and `Batch` / `LibraryType` are confounded with it, so every split below is over patients and the batch-corrected and uncorrected numbers are read as a pair.

In [ ]:
# setup + audit: how much of remission is already explained by technical covariates
import json
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SCLR_REPO  = Path("/ictstr01/groups/ml01/workspace/christopher.lance/SampleCLR")
OUTPUT_DIR = Path("hca-gut-atlas-downstream/data/taurus_data/sampleclr_beyond_batch")
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "representations").mkdir(parents=True, exist_ok=True)

PATIENT_KEY = "Patient"
BATCH_KEY   = "Batch"
TARGET      = "Remission_clean"      # created in the next cell

# colour follows the family of a representation, never its rank
FAMILY_COLORS = {
    "baseline":       "#2a78d6",
    "zero-shot":      "#eb6834",
    "supervised":     "#1baf7a",
    "batch-corrected": "#4a3aa7",
}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

sm = meta_adata.obs
print(f"{len(sm)} samples | {sm[PATIENT_KEY].nunique()} patients | "
      f"median {sm[PATIENT_KEY].value_counts().median():.0f} samples/patient\n")

# 1. remission is a patient-level label
lab = sm["Remission_status"].astype(str).str.strip()
print(pd.DataFrame({
    "samples":  lab.value_counts(),
    "patients": sm.assign(_l=lab).groupby("_l", observed=True)[PATIENT_KEY].nunique(),
}).to_string(), "\n")

n_multi = sm.assign(_l=lab).groupby(PATIENT_KEY, observed=True)["_l"].nunique().gt(1).sum()
print(f"patients with more than one remission value: {int(n_multi)}"
      "  <- 0 means the label carries no within-patient information\n")

# 2. how much of the outcome is already in the technical covariates?
def majority_rule_accuracy(feature, target):
    """Accuracy of 'predict the majority target class within each level of feature'."""
    df = pd.DataFrame({"f": feature.astype(str), "t": target.astype(str)}).dropna()
    pred = df.groupby("f", observed=True)["t"].transform(lambda s: s.mode().iat[0])
    return (pred == df["t"]).mean()

cd = lab.isin(["Remission", "Non_Remission"]).to_numpy()      # drop healthy
print(f"majority-class baseline                 : {lab[cd].value_counts(normalize=True).max():.3f}")
for c in [BATCH_KEY, "LibraryType", "Site", "Ileum_vs_Colon", PATIENT_KEY]:
    print(f"predict remission from {c:<17}: {majority_rule_accuracy(sm.loc[cd, c], lab[cd]):.3f}")

print("\n--- Batch x Remission (samples) ---")
print(pd.crosstab(sm.loc[cd, BATCH_KEY].astype(str), lab[cd]).to_string())
print("\n--- LibraryType x Remission (samples) ---")
print(pd.crosstab(sm.loc[cd, "LibraryType"].astype(str), lab[cd]).to_string())
print("\n--- patients per Batch (Batch is nested inside Patient) ---")
print(sm.groupby(BATCH_KEY, observed=True)[PATIENT_KEY].nunique().to_string())

#### (a) Clean the target

The "no label" level is the literal string `'None '`; mapping it to `NaN` keeps healthy samples in the contrastive loss but out of the classification head.

In [ ]:
# (a) clean the target: 'None ' / 'Not_avail' -> NaN
REMISSION_NA = {"None", "Not_avail", "nan", "NaN", "<NA>", ""}

def clean_remission(s):
    s = s.astype(str).str.strip()
    return s.where(~s.isin(REMISSION_NA), np.nan)

for _ad in (adata_pb, meta_adata):
    _ad.obs[TARGET] = clean_remission(_ad.obs["Remission_status"])
    # drop any stale encoding so SampleCLR re-encodes from the cleaned column
    _ad.obs.drop(columns=[f"{TARGET}_encoded"], errors="ignore", inplace=True)

print("cells   :", adata_pb.obs[TARGET].value_counts(dropna=False).to_dict())
print("samples :", meta_adata.obs[TARGET].value_counts(dropna=False).to_dict())
print("labelled patients:",
      meta_adata.obs.loc[meta_adata.obs[TARGET].notna(), PATIENT_KEY].nunique(),
      "of", meta_adata.obs[PATIENT_KEY].nunique())

#### (b) Patient-grouped train / val / test

Splits are drawn over patients and expanded to sample IDs; `test_ids` is a held-back reserve, the real scoring is the leave-one-patient-out pass below.

In [ ]:
# (b) patient-grouped splits, expanded to sample IDs
def patient_split(seed=0, test_frac=0.25, val_frac=0.25, verbose=True):
    """Split patients (not samples) into train/val/test; returns three lists of sample IDs."""
    obs = meta_adata.obs
    lab_by_pat = (obs.loc[obs[TARGET].notna()]
                     .groupby(PATIENT_KEY, observed=True)[TARGET].first())
    pats, y = lab_by_pat.index.to_numpy(), lab_by_pat.to_numpy()

    rest_i, test_i = next(StratifiedShuffleSplit(
        n_splits=1, test_size=test_frac, random_state=seed).split(pats, y))
    tr_i, val_i = next(StratifiedShuffleSplit(
        n_splits=1, test_size=val_frac, random_state=seed).split(pats[rest_i], y[rest_i]))

    test_pats  = set(pats[test_i])
    val_pats   = set(pats[rest_i][val_i])
    train_pats = (set(pats[rest_i][tr_i])
                  | set(obs.loc[obs[TARGET].isna(), PATIENT_KEY].astype(str).unique()))

    # a patient must live in exactly one split -- this is the leakage guard
    assert not (train_pats & val_pats), train_pats & val_pats
    assert not (train_pats & test_pats), train_pats & test_pats
    assert not (val_pats & test_pats),   val_pats & test_pats

    def ids(p):
        return obs.index[obs[PATIENT_KEY].astype(str).isin({str(x) for x in p})].tolist()

    splits = {"train": (train_pats, ids(train_pats)),
              "val":   (val_pats,   ids(val_pats)),
              "test":  (test_pats,  ids(test_pats))}
    if verbose:
        for name, (p, s) in splits.items():
            print(f"  {name:5s}: {len(p):2d} patients, {len(s):3d} samples  "
                  f"{obs.loc[s, TARGET].value_counts().to_dict()}")
    return [splits[k][1] for k in ("train", "val", "test")]

train_ids, val_ids, test_ids = patient_split(seed=0)
assert not (set(train_ids) & set(val_ids) & set(test_ids))
print(f"\n{len(train_ids) + len(val_ids) + len(test_ids)} of {meta_adata.n_obs} samples assigned")

#### (c) Model config + ablation grid

Grid over supervision (none / remission / inflammation), batch-aware sampling and `lambda_`, plus derived arms with `Inflammation_score` projected out post-hoc.

In [ ]:
# (c) model config + ablation grid
SCLR_SIZE = "tiny"                       # "tiny" | "medium" | "large"
sclr_cfg  = json.loads((SCLR_REPO / f"configs/models/{SCLR_SIZE}.json").read_text())
sclr_cfg["contrastive_loss_temperature"] = 0.1
sclr_cfg.pop("batch_size", None)                      # passed explicitly below
print(f"{SCLR_SIZE}.json ->", json.dumps(sclr_cfg, indent=2))

CELL_SELECTION   = "random"              # 'eigenvector' needs cugraph
BASE_TASKS       = {"classification": [TARGET]}
INFLAM_KEY       = "Inflammation_score"  # continuous histology score, one value per sample
INFLAM_TASKS     = {"regression": [INFLAM_KEY]}
LAMBDA_DEFAULT   = 1.0
RUN_LAMBDA_SWEEP = True
LAMBDA_SWEEP     = [0.5, 1.0, 3.0, 10.0]
N_SEEDS          = 1                     # raise to average over split draws

CONFIGS = [
    dict(rep_name="sclr_zeroshot",             tasks=None,         batch_aware=False, lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_zeroshot_batchaware",  tasks=None,         batch_aware=True,  lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_remission",            tasks=BASE_TASKS,   batch_aware=False, lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_remission_batchaware", tasks=BASE_TASKS,   batch_aware=True,  lambda_=LAMBDA_DEFAULT),
    dict(rep_name="sclr_remission_batchcorr",  batch_aware=False,  lambda_=LAMBDA_DEFAULT,
         tasks={**BASE_TASKS, "batch_correction": BATCH_KEY}),
    # inflammation severity alone: a biopsy-level, within-patient axis, orthogonal to remission
    dict(rep_name="sclr_inflammation",         tasks=INFLAM_TASKS, batch_aware=False, lambda_=LAMBDA_DEFAULT),
]
if RUN_LAMBDA_SWEEP:
    CONFIGS += [dict(rep_name=f"sclr_remission_lam{l:g}".replace(".", "p"),
                     tasks=BASE_TASKS, batch_aware=False, lambda_=l)
                for l in LAMBDA_SWEEP if l != LAMBDA_DEFAULT]

# derived arms: Inflammation_score projected out post-hoc, baselines included as the reference
RESIDUALIZE_ON = INFLAM_KEY
RESIDUALIZE    = ["pseudobulk", "composition",
                  "sclr_remission", "sclr_remission_batchaware", "sclr_remission_batchcorr"]
RESID_SUFFIX   = "_inflamresid"

# residualisation is a secondary encoding (marker shape / hatch); an arm keeps its parent's colour
def is_residualised(name):
    return RESID_SUFFIX in name

def rep_family(name):
    n = name.lower().replace(RESID_SUFFIX, "")          # residualised keeps its parent's family
    if not n.startswith("sclr"):                        return "baseline"
    if "batchcorr" in n or "batchaware" in n:           return "batch-corrected"
    if "zeroshot" in n:                                 return "zero-shot"
    return "supervised"

print(f"\n{len(CONFIGS)} configs x {N_SEEDS} seed(s) = {len(CONFIGS) * N_SEEDS} models")
for c in CONFIGS:
    print(f"  {c['rep_name']:34s} tasks={str(c['tasks']):55s} "
          f"batch_aware={c['batch_aware']!s:5s} lambda={c['lambda_']}")
print(f"\n+ {len(RESIDUALIZE)} derived arms (no training): {RESIDUALIZE_ON} projected out of")
for b in RESIDUALIZE:
    print(f"  {b}{RESID_SUFFIX}")

In [ ]:
# train the grid, skipping anything already cached on disk
import contextlib, io, hashlib

subset_size = int(min(counts.loc[keep_samples].max(), 512))
bb_reps = []
splits_by_seed = {}          # seed -> train sample IDs, so residualisation uses the right split

def cfg_fingerprint(cfg, seed):
    """Hash of everything that changes the model, so a stale cached representation is caught."""
    payload = json.dumps({"cfg": cfg, "seed": seed, "sclr_cfg": sclr_cfg,
                          "layer": LAYER, "min_cells": MIN_CELLS},
                         sort_keys=True, default=str)
    return hashlib.sha1(payload.encode()).hexdigest()[:12]

for seed in range(N_SEEDS):
    tr, va, te = patient_split(seed=seed, verbose=(seed == 0))
    splits_by_seed[seed] = tr

    for cfg in CONFIGS:
        rep_name  = f"{cfg['rep_name']}_seed{seed}"
        rep_path  = OUTPUT_DIR / "representations" / f"{rep_name}_representations.npy"
        ckpt_path = OUTPUT_DIR / "models" / f"{rep_name}.pt"
        fingerprint = cfg_fingerprint(cfg, seed)

        if rep_path.exists():
            cached_fp = (torch.load(ckpt_path, map_location="cpu", weights_only=False).get("fingerprint")
                         if ckpt_path.exists() else None)
            if cached_fp is not None and cached_fp != fingerprint:
                print(f"!! {rep_name}: cached representation was trained under a different config "
                      f"({cached_fp} != {fingerprint}). Delete {rep_path.name} to retrain; "
                      f"loading the stale one for now.")
            store_rep(rep_name, np.load(rep_path), dist=None, samples=list(meta_adata.obs_names))
            bb_reps.append(rep_name)
            continue

        print(f"\n{'=' * 78}\n{rep_name}  |  tasks={cfg['tasks']}  "
              f"batch_aware={cfg['batch_aware']}  lambda_={cfg['lambda_']}\n{'=' * 78}")

        kw = dict(
            adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
            train_ids=tr, val_ids=va, test_ids=te,
            supervised_class_balancing="inverse",     # 68 Remission vs 28 Non_Remission samples
            lambda_=cfg["lambda_"],
            n_cells_per_sample=[50, 150], batch_size=32,
            early_stopping_patience=50,
            num_warmup_epochs_stage1=10, num_warmup_epochs_stage2=10,
            device=device, seed=seed, verbose=False, **sclr_cfg,
        )
        if cfg["tasks"] is not None:
            kw["tasks"] = cfg["tasks"]
        if cfg["batch_aware"]:
            kw.update(use_batch_aware_sampler=True,
                      batch_sampler_batch_col=BATCH_KEY,
                      # Batch_4 has only 5 samples -- merge undersized batches for the sampler
                      batch_sampler_pseudo_batch_strategy="similarity")

        model = sampleclr.ContrastiveModel(**kw)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            model.train(num_epochs_stage1=100, num_epochs_stage2=150,
                        stage1_val_metric="loss",
                        stage2_val_metric="total" if cfg["tasks"] else "loss",
                        verbose=False)

        reps_arr = get_sample_representations_from_adata(
            projector=model.projector, aggregator=model.aggregator,
            adata=adata_pb, sample_key=SAMPLE_KEY, layer=LAYER,
            meta_obs_names=list(meta_adata.obs_names), subset_size=subset_size,
            device=device, cell_selection=CELL_SELECTION)

        store_rep(rep_name, reps_arr, dist=None, samples=list(meta_adata.obs_names))
        bb_reps.append(rep_name)

        np.save(rep_path, reps_arr)
        torch.save({
            "projector":  model.projector.state_dict(),
            "aggregator": model.aggregator.state_dict(),
            # every head group is saved, so load_sclr_model can restore any arm
            "classifiers":        [c.state_dict() for c in getattr(model, "classifiers", [])],
            "regressors":         [r.state_dict() for r in getattr(model, "regressors", [])],
            "ordinal_regressors": [o.state_dict() for o in getattr(model, "ordinal_regressors", [])],
            "cfg": cfg, "seed": seed, "layer": model.layer, "output_dim": model.output_dim,
            "sclr_cfg": sclr_cfg, "fingerprint": fingerprint,
            "split": {"train": tr, "val": va, "test": te},
        }, ckpt_path)

meta_adata.obs["split"] = "train"
meta_adata.obs.loc[val_ids,  "split"] = "val"
meta_adata.obs.loc[test_ids, "split"] = "test"
print("\ntrained/loaded:", bb_reps)

# derived arms: project Inflammation_score out of an existing representation
def residualise(X, values, fit_mask):
    """OLS removal of the linear `values` component, with coefficients fit on train patients only."""
    X = np.asarray(X, dtype=np.float64)
    v = np.asarray(values, dtype=np.float64)
    D = np.column_stack([np.ones(len(v)), v])
    ok = fit_mask & np.isfinite(v) & np.isfinite(X).all(axis=1)
    coef, *_ = np.linalg.lstsq(D[ok], X[ok], rcond=None)
    return (X - D @ coef).astype(np.float32)

_score = pd.to_numeric(meta_adata.obs[RESIDUALIZE_ON], errors="coerce").to_numpy(float)
assert np.isfinite(_score).all(), \
    f"{RESIDUALIZE_ON} has {int(np.isnan(_score).sum())} NaN; the projection would be undefined there"
print(f"\nprojecting out {RESIDUALIZE_ON} (coefficients fit on train patients, applied to all "
      f"{len(_score)} samples)")

resid_reps = []
for base in RESIDUALIZE:
    # a seedless name means a patpy baseline (one copy); an sclr arm has one per seed
    names = ([base] if f"{base}_representations" in meta_adata.obsm
             else [f"{base}_seed{s}" for s in range(N_SEEDS)])
    for name in names:
        if f"{name}_representations" not in meta_adata.obsm:
            print(f"  skip {name}: no representation stored")
            continue
        # baselines carry no seed, so they are residualised on the seed-0 split
        seed = int(name.rsplit("_seed", 1)[1]) if "_seed" in name else 0
        fit  = meta_adata.obs_names.isin(splits_by_seed.get(seed, train_ids))
        out  = f"{name}{RESID_SUFFIX}"
        store_rep(out, residualise(meta_adata.obsm[f"{name}_representations"], _score, fit),
                  dist=None, samples=list(meta_adata.obs_names))
        resid_reps.append(out)

bb_reps += resid_reps
print("\nderived:", resid_reps)

#### (d) Benchmark, with `Patient` as the leakage detector

kNN recovery per covariate (`f1_macro_calibrated`: 0 = chance, 1 = perfect), with `Patient` scored uninverted so remission recovery that is only patient re-identification stays visible.

In [ ]:
# (d) benchmark: covariate recovery, with Patient as the leakage detector
bb_baselines = [m for m in ["pseudobulk", "composition", "gloscope", "CT_pseudobulk"]
                if f"{m}_representations" in meta_adata.obsm]
bb_methods   = bb_baselines + bb_reps
print("scoring:", bb_methods)

benchmark_schema_bb = {
    "relevant": {
        TARGET:         "classification",
        "Treatment":    "classification",
        "Inflammation": "classification",
        INFLAM_KEY:     "regression",
    },
    "technical": {                          # reported inverted: higher = less leakage
        PATIENT_KEY:   "classification",
        BATCH_KEY:     "classification",
        "LibraryType": "classification",
        "n_cells":     "regression",
    },
    "contextual": {
        "Ileum_vs_Colon": "classification",
        "Site":           "classification",
    },
}

# same covariates under "relevant" -> RAW recovery, which is what a diagnostic needs
leak_schema = {"relevant": {PATIENT_KEY:   "classification",
                            BATCH_KEY:     "classification",
                            "LibraryType": "classification",
                            TARGET:        "classification",
                            INFLAM_KEY:    "regression"}}

bb_scores = pd.concat(
    [pat.tl.evaluation.knn_prediction_score(meta_adata, benchmark_schema_bb, representations=[m])
     for m in bb_methods], ignore_index=True)
bb_leak = pd.concat(
    [pat.tl.evaluation.knn_prediction_score(meta_adata, leak_schema, representations=[m])
     for m in bb_methods], ignore_index=True)

bb_wide   = bb_scores.pivot_table(index="representation", columns="covariate", values="score")
leak_wide = bb_leak.pivot_table(index="representation", columns="covariate", values="score")
leak_wide = leak_wide.loc[bb_methods]        # keep grid order, not alphabetical

print("\n=== uninverted kNN recovery, f1_macro_calibrated: 0 = chance, 1 = perfect ===")
display(leak_wide[[TARGET, PATIENT_KEY, BATCH_KEY, "LibraryType"]]
        .rename(columns={TARGET: "Remission"})
        .style.format("{:.3f}").background_gradient(cmap="PiYG", vmin=0, vmax=1))

# did the projection remove Inflammation_score, and what did it cost? (regression -> signed spearman r)
_pairs = [(b, f"{b}{RESID_SUFFIX}") for b in leak_wide.index if f"{b}{RESID_SUFFIX}" in leak_wide.index]
if _pairs:
    removal = pd.DataFrame(
        [{"representation":        base,
          f"{INFLAM_KEY} (before)": leak_wide.loc[base, INFLAM_KEY],
          f"{INFLAM_KEY} (after)":  leak_wide.loc[res,  INFLAM_KEY],
          "Remission (before)":     leak_wide.loc[base, TARGET],
          "Remission (after)":      leak_wide.loc[res,  TARGET],
          PATIENT_KEY:              leak_wide.loc[res,  PATIENT_KEY]}
         for base, res in _pairs]).set_index("representation")
    removal["Remission cost"] = removal["Remission (after)"] - removal["Remission (before)"]

    print(f"\n=== {RESIDUALIZE_ON} removal check ===")
    print("left pair : spearman_r. Should collapse in magnitude. A small negative value is")
    print("            expected, not a bug -- the projection is fit on train patients only,")
    print("            so held-out samples end up slightly over-corrected.")
    print("right pair: f1_macro_calibrated, the price paid in remission recovery.")
    display(removal.style.format("{:+.3f}")
            .background_gradient(cmap="PuOr", subset=[f"{INFLAM_KEY} (before)",
                                                      f"{INFLAM_KEY} (after)"], vmin=-1, vmax=1)
            .background_gradient(cmap="PiYG", subset=["Remission cost"], vmin=-0.5, vmax=0.5))
else:
    print(f"\nno {RESID_SUFFIX} arms stored -- run the training cell first")

In [ ]:
%matplotlib inline
# does the Remission score track patient leakage? signal lives above the diagonal
fig, ax = plt.subplots(figsize=(7.2, 6.4))
ax.set_facecolor("none")

lim = max(0.35, leak_wide[[TARGET, PATIENT_KEY]].to_numpy().max() * 1.25)
ax.plot([0, lim], [0, lim], color=GRID, lw=2, zorder=1)
ax.annotate("remission = patient identity", xy=(lim * 0.62, lim * 0.62),
            xytext=(6, -14), textcoords="offset points", color=MUTED, fontsize=9)

# connector from each base arm to its residualised twin
for base in leak_wide.index:
    res = f"{base}{RESID_SUFFIX}"
    if res in leak_wide.index:
        ax.plot(leak_wide.loc[[base, res], PATIENT_KEY], leak_wide.loc[[base, res], TARGET],
                color=FAMILY_COLORS[rep_family(base)], lw=1.2, alpha=0.55, zorder=2)

seen = set()
for rep in leak_wide.index:
    fam  = rep_family(rep)
    resid = is_residualised(rep)
    x, y = leak_wide.loc[rep, PATIENT_KEY], leak_wide.loc[rep, TARGET]
    ax.scatter(x, y, s=110, color=FAMILY_COLORS[fam], marker="D" if resid else "o",
               edgecolor="white", linewidth=2, zorder=3, label=fam if fam not in seen else None)
    seen.add(fam)
    ax.annotate(rep.replace("_seed0", "").replace("sclr_", "").replace(RESID_SUFFIX, " -infl"),
                xy=(x, y), xytext=(8, -3), textcoords="offset points", fontsize=9, color=INK)

ax.set_xlabel("Patient recovery  (f1_macro_calibrated, 0 = chance)", color=INK)
ax.set_ylabel("Remission recovery  (f1_macro_calibrated, 0 = chance)", color=INK)
ax.set_title("Remission signal vs patient leakage\nabove the diagonal = beyond patient identity",
             color=INK, loc="left")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.grid(True, color=GRID, lw=0.8, zorder=0)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color("#c3c2b7")
ax.tick_params(colors=MUTED)

# two legends: colour = family, shape = whether Inflammation_score was projected out
leg_fam = ax.legend(frameon=False, loc="upper right", labelcolor=INK)
ax.add_artist(leg_fam)
shape_handles = [plt.Line2D([], [], marker=m, ls="", ms=9, color=MUTED, mec="white", mew=1.5, label=l)
                 for m, l in [("o", "as trained"), ("D", f"{RESIDUALIZE_ON} projected out")]]
ax.legend(handles=shape_handles, frameon=False, loc="lower right", labelcolor=INK, fontsize=9)
plt.tight_layout(); plt.show()

#### (e) Patient-level leave-one-patient-out evaluation

Sample embeddings pooled per patient, then leave-one-patient-out logistic regression; one-hot `Batch` goes through the same loop as the reference to beat.

In [ ]:
# (e) leave-one-patient-out at the patient level
_lab_ok = meta_adata.obs[TARGET].notna().to_numpy()
_pat    = meta_adata.obs[PATIENT_KEY].astype(str).to_numpy()[_lab_ok]
_y      = (meta_adata.obs[TARGET].to_numpy()[_lab_ok] == "Remission").astype(int)

y_pat = pd.Series(_y, index=_pat).groupby(level=0).first()
print(f"{len(y_pat)} labelled patients: "
      f"{int(y_pat.sum())} Remission / {int((1 - y_pat).sum())} Non_Remission")

def to_patient_matrix(X):
    """Mean-pool sample embeddings within each patient; rows aligned to y_pat."""
    return pd.DataFrame(np.asarray(X)[_lab_ok], index=_pat).groupby(level=0).mean().loc[y_pat.index]

def lopo_scores(Xp, yp, C=1.0):
    """Leave-one-patient-out CV; returns (auc, balanced_accuracy) over pooled predictions."""
    prob = np.full(len(Xp), np.nan)
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    for tr, te in LeaveOneGroupOut().split(Xp, yp, groups=np.asarray(Xp.index)):
        clf.fit(Xp.iloc[tr], yp.iloc[tr])
        prob[te] = clf.predict_proba(Xp.iloc[te])[:, 1]
    return roc_auc_score(yp, prob), balanced_accuracy_score(yp, (prob >= 0.5).astype(int))

rows = []
for m in bb_methods:
    auc, bacc = lopo_scores(to_patient_matrix(meta_adata.obsm[f"{m}_representations"]), y_pat)
    rows.append({"representation": m, "family": rep_family(m), "AUC": auc, "balanced_acc": bacc})

# technical-covariate references, through the identical LOPO loop
for col in [BATCH_KEY, "LibraryType"]:
    oh = pd.get_dummies(meta_adata.obs[col].astype(str)).to_numpy(dtype=float)
    auc, bacc = lopo_scores(to_patient_matrix(oh), y_pat)
    rows.append({"representation": f"reference: {col} (one-hot)", "family": "reference",
                 "AUC": auc, "balanced_acc": bacc})
rows.append({"representation": "reference: chance", "family": "reference",
             "AUC": 0.5, "balanced_acc": 0.5})

lopo = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
BATCH_REF = float(lopo.loc[lopo["representation"] == f"reference: {BATCH_KEY} (one-hot)", "AUC"].iat[0])
display(lopo.style.format({"AUC": "{:.3f}", "balanced_acc": "{:.3f}"})
        .background_gradient(cmap="PiYG", subset=["AUC"], vmin=0.3, vmax=1.0))
print(f"\nreference to beat -- {BATCH_KEY} alone: AUC {BATCH_REF:.3f}")
print("clears it:", lopo.loc[(lopo.family != "reference") & (lopo.AUC > BATCH_REF),
                             "representation"].tolist() or "none")

In [ ]:
%matplotlib inline
import matplotlib.patches as mpatches
# patient-level LOPO AUC against the technical references (hatched = Inflammation_score removed)
FAM_COLORS_ALL = {**FAMILY_COLORS, "reference": MUTED}

plot_df = lopo.sort_values("AUC")            # ascending -> best on top of an hbar
fig, ax = plt.subplots(figsize=(8.4, 0.42 * len(plot_df) + 1.8))
ax.set_facecolor("none")

bars = ax.barh(plot_df["representation"].str.replace("_seed0", "", regex=False),
               plot_df["AUC"],
               color=[FAM_COLORS_ALL[f] for f in plot_df["family"]],
               hatch=["//" if is_residualised(r) else "" for r in plot_df["representation"]],
               edgecolor="white", linewidth=0,
               height=0.62, zorder=3)

ax.set_ylim(-0.6, len(plot_df) - 0.5 + 0.95)
ax.axvline(0.5, color=GRID, lw=2, zorder=1)
ax.axvline(BATCH_REF, color=MUTED, lw=2, ls=(0, (4, 3)), zorder=2)
ax.annotate(f"{BATCH_KEY} alone ({BATCH_REF:.2f})", xy=(BATCH_REF, len(plot_df) - 0.1),
            xytext=(5, 0), textcoords="offset points",
            color=MUTED, fontsize=9, va="center")

for b, v in zip(bars, plot_df["AUC"]):
    ax.text(v + 0.008, b.get_y() + b.get_height() / 2, f"{v:.2f}",
            va="center", fontsize=9, color=INK)

ax.set_xlim(0, 1.06)
ax.set_xlabel("Leave-one-patient-out AUC  (16 patients)", color=INK)
ax.set_title("Remission prediction from patient-pooled sample embeddings",
             color=INK, loc="left")
ax.grid(True, axis="x", color=GRID, lw=0.8, zorder=0)
ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(colors=MUTED)
ax.tick_params(axis="y", length=0, labelcolor=INK)

handles = [plt.Line2D([], [], marker="s", ls="", ms=9, color=c, label=f)
           for f, c in FAM_COLORS_ALL.items() if f in set(plot_df["family"])]
if plot_df["representation"].map(is_residualised).any():
    handles.append(mpatches.Patch(facecolor=MUTED, hatch="//", edgecolor="white",
                                  label=f"{RESIDUALIZE_ON} projected out"))
ax.legend(handles=handles, frameon=False, loc="lower right", labelcolor=INK)
plt.tight_layout(); plt.show()

# patient_f1_cal is the leakage column: a high AUC next to a high one is not a result
lopo.join(leak_wide[[PATIENT_KEY]].rename(columns={PATIENT_KEY: "patient_f1_cal"}),
          on="representation")

#### (f) The inflammation axis, and what removing it costs

Leave-one-patient-out ridge on `Inflammation_score` at the sample level; `r_within_patient` is the honest column, and the `*_inflamresid` arms should collapse on it without losing remission AUC.

In [ ]:
# (f) leave-one-patient-out ridge on Inflammation_score, at the sample level
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr

_score_all = pd.to_numeric(meta_adata.obs[INFLAM_KEY], errors="coerce")
_pat_all   = meta_adata.obs[PATIENT_KEY].astype(str)
_ok        = _score_all.notna().to_numpy()
print(f"{int(_ok.sum())} scored samples from {_pat_all[_ok].nunique()} patients | "
      f"score {_score_all[_ok].min():.2f}-{_score_all[_ok].max():.2f}")

def lopo_regression(X, alpha=1.0):
    """Returns (pooled r, within-patient r); the second centres both on the patient mean first."""
    X = np.asarray(X, dtype=np.float64)[_ok]
    y = _score_all.to_numpy(float)[_ok]
    g = _pat_all.to_numpy()[_ok]
    pred = np.full(len(y), np.nan)
    reg = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    for tr_i, te_i in LeaveOneGroupOut().split(X, y, groups=g):
        reg.fit(X[tr_i], y[tr_i])
        pred[te_i] = reg.predict(X[te_i])

    pooled = spearmanr(y, pred).statistic
    df = pd.DataFrame({"y": y, "p": pred, "g": g})
    multi = df.groupby("g")["y"].transform("size") >= 2
    df = df[multi]
    if df["g"].nunique() < 2:
        return pooled, np.nan
    dy = df["y"] - df.groupby("g")["y"].transform("mean")
    dp = df["p"] - df.groupby("g")["p"].transform("mean")
    return pooled, spearmanr(dy, dp).statistic

rows = []
for m in bb_methods:
    r_pool, r_within = lopo_regression(meta_adata.obsm[f"{m}_representations"])
    rows.append({"representation": m, "family": rep_family(m),
                 "residualised": is_residualised(m),
                 "r_pooled": r_pool, "r_within_patient": r_within})

# references through the identical loop
for col in ["Ileum_vs_Colon", BATCH_KEY]:
    oh = pd.get_dummies(meta_adata.obs[col].astype(str)).to_numpy(dtype=float)
    r_pool, r_within = lopo_regression(oh)
    rows.append({"representation": f"reference: {col} (one-hot)", "family": "reference",
                 "residualised": False, "r_pooled": r_pool, "r_within_patient": r_within})

inflam = pd.DataFrame(rows).sort_values("r_within_patient", ascending=False).reset_index(drop=True)
REGION_REF = float(inflam.loc[inflam["representation"] == "reference: Ileum_vs_Colon (one-hot)",
                              "r_pooled"].iat[0])

print(f"\n=== LOPO ridge on {INFLAM_KEY} (sample level, split by patient) ===")
print("r_pooled is inflated by anatomy; r_within_patient is the honest column.")
display(inflam.style.format({"r_pooled": "{:+.3f}", "r_within_patient": "{:+.3f}"})
        .background_gradient(cmap="PuOr", subset=["r_pooled", "r_within_patient"],
                             vmin=-0.8, vmax=0.8))
print(f"reference -- region alone explains r_pooled = {REGION_REF:+.3f}")

_res_rows = inflam[inflam["residualised"]]
if len(_res_rows):
    print(f"\nprojection check -- {RESIDUALIZE_ON} arms should sit near 0 on both columns:")
    print(_res_rows[["representation", "r_pooled", "r_within_patient"]]
          .round(3).to_string(index=False))
    print(f"\nworst residual leakage: |r_within_patient| = "
          f"{_res_rows['r_within_patient'].abs().max():.3f}")

## Benchmark and plots

kNN over each sample-distance matrix: high on `relevant` is good, high on `technical` means the representation leaks batch.

In [ ]:
%matplotlib inline
benchmark_schema = {
    "relevant": {                        # biology we want the sample embedding to capture
        "Disease":            "classification",
        "Inflammation":       "classification",
        "Inflammation_score": "regression",
        "Treatment":          "classification",
        "Remission_status":   "classification"
    },
    "technical": {                       # nuisance; scored reversed, so higher = better
        "Batch":       "classification",
        "LibraryType": "classification",
        "n_cells":     "regression",
    },
    "contextual": {                      # fine to catch, but not the target
        "Site":             "classification",
        "Ileum_vs_Colon":   "classification",
        "Disease_duration": "regression",
        "Age":              "regression",
        "Gender":           "classification",
    },
}
methods = list(meta_adata.uns["sample_representations"])
patpy_all = pd.concat(
    [pat.tl.evaluation.knn_prediction_score(meta_adata, benchmark_schema, representations=[m])
     for m in methods],
    ignore_index=True,
)
knn_wide = patpy_all.pivot_table(index="representation", columns="covariate", values="score")
knn_wide

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=patpy_all, y="covariate", x="score", hue="representation",
            orient="h", palette="tab10", ax=ax)
ax.set_xlim(0, 1.02)
ax.set_title("kNN recovery of covariates from sample distances\n(relevant: higher = better | technical: lower = better)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# define color scheme

In [ ]:
fig, ax = plt.subplots(figsize=(25, 15))

Table(knn_results_wide[cols_order].sort_values("total", ascending=False), column_definitions=tuple(col_defs), ax=ax)
#fig.savefig("patpy_results_table_steph_comb.png", dpi=300, bbox_inches="tight")

### Visualise the sample-level embedding

One UMAP over samples per representation, coloured by the sample-level covariates.

In [ ]:
sc.settings.figdir = 'hca-gut-atlas-tutorial/images/temp_images/'
sc.settings.set_figure_params(dpi=180, format='svg', transparent=True)

In [ ]:
%matplotlib inline
COLORS = [
    "Disease", "Inflammation", "Inflammation_score", "Treatment",
    "Remission_status","Batch","LibraryType","n_cells", "Site"
    ]
for rep in ["pseudobulk", "composition", "gloscope", "sampleclr_remission",
            "sclr_remission_lam3_seed0", "sclr_remission_lam10_seed0",
            "sclr_remission_batchaware_seed0", 
            "sclr_inflammation_seed0",                    # the inflammation axis
            "sclr_remission_seed0_inflamresid"]:          # ...projected back out
    if f"{rep}_representations" not in meta_adata.obsm:
        continue
    ehr.pp.neighbors(meta_adata, use_rep=f"{rep}_representations", key_added=f"{rep}_nn", n_neighbors=15)
    ehr.tl.umap(meta_adata, neighbors_key=f"{rep}_nn")
    meta_adata.obsm[f"{rep}_umap"] = meta_adata.obsm["X_umap"].copy()
    ehr.pl.umap(meta_adata, color=COLORS, ncols=3, size=200, wspace=0.4,
                frameon=False,
                title=[f"{rep} · {c}" for c in COLORS], save=f"_{rep}.svg")

## Cell type gene - batch aware

In [ ]:
# the embedding to interpret: any rep_name trained above
INTERP_REP = "sclr_remission_batchaware_seed0"

import sys, gc, inspect, json
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc, torch
import matplotlib.pyplot as plt, seaborn as sns

sys.path.insert(0, "/lustre/groups/ml01/workspace/christopher.lance/hca_gut/hca-gut-atlas-tutorial/temp_ref_code")
import interpretation_utils as iu
from interpretation_utils import (
    get_cell_weights, precompute_aggregations, correlate_and_rank,
    run_interpretation_pipeline, load_interp_dict, correct_pvalues_matrix,
    plot_head_top_genes, plot_gene_head_scatter,
)

# keys
CT_KEY       = "predictions"          # fine annotation, 91 levels
LINEAGE_KEY  = "predicted_lineage"    # coarse alternative, 4 levels
INTERP_CT_KEY = CT_KEY
COVARIATE    = TARGET
COV_POSITIVE = "Remission"            # level scored as 1

SCLR_DIR   = Path("hca-gut-atlas-downstream/data/taurus_data/sampleclr_beyond_batch")
INTERP_DIR = SCLR_DIR / "interpretation" / INTERP_REP
(INTERP_DIR / "rankings").mkdir(parents=True, exist_ok=True)

ckpt_path = SCLR_DIR / "models" / f"{INTERP_REP}.pt"
assert ckpt_path.exists(), f"no checkpoint at {ckpt_path}\navailable: " \
    f"{sorted(p.stem for p in (SCLR_DIR / 'models').glob('*.pt'))}"

_state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"interpreting : {INTERP_REP}")
print(f"  tasks      : {_state['cfg']['tasks']}")
print(f"  lambda_    : {_state['cfg']['lambda_']}   batch_aware: {_state['cfg']['batch_aware']}")
print(f"  layer      : {_state['layer']}   heads: {_state['sclr_cfg']['n_aggregator_heads']}")
print(f"  output dir : {INTERP_DIR}")

In [ ]:
# cell-level object for interpretation: attention is correlated against gene expression, so adata_pb (0 features) cannot be reused
if "adata_interp" not in globals():
    _mask = adata_taurus_CD.obs[SAMPLE_KEY].isin(keep_samples).to_numpy()
    adata_interp = adata_taurus_CD[_mask].copy()
    adata_interp.X = adata_interp.X.astype(np.float32)      # float64 -> half the memory in to_array()

    # CP10K + log1p, so per-sample mean expression does not just track sequencing depth
    sc.pp.normalize_total(adata_interp, target_sum=1e4)
    sc.pp.log1p(adata_interp)

    # var_names are Ensembl IDs; the rankings are unreadable without symbols
    _sym = adata_interp.var["gene_symbol"].astype(str)
    adata_interp.var["ensembl_id"] = adata_interp.var_names
    adata_interp.var_names = np.where(_sym.isin(["", "nan", "None"]),
                                      adata_interp.var_names, _sym)
    adata_interp.var_names_make_unique()

    # carry the cleaned label over so tasks/heads resolve and covariates are available
    for _c in (TARGET, PATIENT_KEY, BATCH_KEY):
        adata_interp.obs[_c] = adata_pb.obs[_c].reindex(adata_interp.obs_names).values
    adata_interp.obs[INTERP_CT_KEY] = adata_interp.obs[INTERP_CT_KEY].astype("category")

_x = adata_interp.X[:500]
print("adata_interp:", adata_interp.shape,
      f"| {adata_interp.obs[SAMPLE_KEY].nunique()} samples"
      f" | {adata_interp.obs[INTERP_CT_KEY].nunique()} levels of {INTERP_CT_KEY}")
print("X after norm+log1p: max %.2f (log scale, so << raw counts)" % _x.max())
print("cells per cell type -- smallest 5:")
print(adata_interp.obs[INTERP_CT_KEY].value_counts().tail(5).to_string())

In [ ]:
# rebuild a trained model from its checkpoint (config in state["cfg"], architecture in state["sclr_cfg"])
def load_sclr_model(rep_name, adata, device=None):
    """Rebuild a ContrastiveModel from a 'Signal beyond Batch' checkpoint and load its weights."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    state  = torch.load(SCLR_DIR / "models" / f"{rep_name}.pt", map_location=device, weights_only=False)

    valid  = set(inspect.signature(sampleclr.ContrastiveModel.__init__).parameters)
    kwargs = {k: v for k, v in state["sclr_cfg"].items() if k in valid}
    if state["cfg"].get("tasks"):
        kwargs["tasks"] = state["cfg"]["tasks"]
    kwargs["lambda_"] = state["cfg"]["lambda_"]

    model = sampleclr.ContrastiveModel(
        adata=adata, sample_key=SAMPLE_KEY, layer=state["layer"],
        train_ids=state["split"]["train"], val_ids=state["split"]["val"],
        device=device, verbose=False, **kwargs,
    )
    # interpretation uses ONLY the aggregator (attention) -- load it strictly.
    model.aggregator.load_state_dict(state["aggregator"])
    model.projector.load_state_dict(state["projector"])
    for group in ("classifiers", "regressors", "ordinal_regressors"):
        try:
            for head, sd in zip(getattr(model, group, []), state.get(group, [])):
                head.load_state_dict(sd)
        except (RuntimeError, ValueError) as e:      # heads are not used below
            print(f"  note: could not restore {group} ({e}); attention is unaffected")
    model.aggregator.eval()
    return model

model = load_sclr_model(INTERP_REP, adata_interp)
N_HEADS = model.n_aggregator_heads
print(f"loaded {INTERP_REP}: {N_HEADS} attention heads, "
      f"aggregator={model.aggregator_type}/{model.aggregator_activation}")

### Per-cell attention

One aggregator pass per sample gives `obsm["X_cell_weights"]` (cells x heads); the summary flags dead heads (all ~ 0) and flat heads (no cell-type preference).

In [ ]:
# per-cell attention: one aggregator pass per sample, exactly as in training
_dev = next(model.aggregator.parameters()).device
_W = np.zeros((adata_interp.n_obs, N_HEADS), dtype=np.float32)

model.aggregator.return_weights = True
with torch.no_grad():
    for sid in adata_interp.obs[SAMPLE_KEY].unique():
        idx = np.where((adata_interp.obs[SAMPLE_KEY] == sid).to_numpy())[0]
        cells = torch.as_tensor(np.asarray(adata_interp.obsm[LAYER][idx]),
                                dtype=torch.float32, device=_dev).unsqueeze(0)
        _, w = model.aggregator(cells, return_weights=True)
        _W[idx] = w.squeeze(0).cpu().numpy()
model.aggregator.return_weights = False

adata_interp.obs.drop(columns=[c for c in adata_interp.obs.columns if c.startswith("head_")],
                      inplace=True, errors="ignore")
adata_interp.obsm["X_cell_weights"] = _W
for i in range(N_HEADS):
    adata_interp.obs[f"head_{i}"] = _W[:, i]

# flag flat heads (pool uniformly) and dead heads (~0 everywhere) before reading any heatmap
summary = pd.DataFrame({
    "mean":         _W.mean(0),
    "max":          _W.max(0),
    "frac_nonzero": (_W > 0).mean(0),
    "selectivity":  _W.std(0) / (_W.mean(0) + 1e-12),   # spread / mean; ~0 = pools everything equally
}, index=[f"head_{i}" for i in range(N_HEADS)])

_norm = model.aggregator_activation == "softmax"
print(f"attention weights: {_W.shape} | activation={model.aggregator_activation} -> "
      + ("normalised across cells within a sample (shares, sum to 1)"
         if _norm else "unnormalised per-cell scores (do NOT sum to 1)"))
print(summary.round(4).to_string())
_dead = summary.index[summary["frac_nonzero"] < 0.01].tolist()
_flat = summary.index[summary["selectivity"] < 0.1].tolist()
print(f"\ndead heads (nothing to interpret): {_dead or 'none'}")
print(f"flat heads (no cell-type preference): {_flat or 'none'}")

In [ ]:
%matplotlib inline
# which cell types does each head pool? colour = mean attention, dot size = fraction non-zero
TOP_CT_PLOT = 30      # 91 levels is unreadable; show the most abundant
_cts = adata_interp.obs[INTERP_CT_KEY].value_counts().head(TOP_CT_PLOT).index.tolist()
_sub = adata_interp[adata_interp.obs[INTERP_CT_KEY].isin(_cts)].copy()
_sub.obs[INTERP_CT_KEY] = _sub.obs[INTERP_CT_KEY].cat.remove_unused_categories()

sc.tl.dendrogram(_sub, groupby=INTERP_CT_KEY, use_rep="X_cell_weights")
with plt.rc_context({"figure.dpi": 130}):
    sc.pl.dotplot(
        _sub, [f"head_{i}" for i in range(N_HEADS)], groupby=INTERP_CT_KEY,
        dendrogram=True, size_title="Fraction of cells with\nnon-zero weight (%)",
        colorbar_title="Mean attention", title=f"{INTERP_REP} — attention by cell type",
    )
del _sub; gc.collect()

### Aggregate per (sample, cell type) and rank genes

Per cell type: mean expression and mean attention per sample, then Spearman-correlate every gene against every active head across samples. Cached; pass `force=True` to recompute.

In [ ]:
# aggregate + rank genes vs attention (the expensive, cached step)
N_INTERP_CT  = None          # most abundant cell types; None = all levels of INTERP_CT_KEY
MIN_SAMPLES  = 20            # a cell type needs this many samples for a usable Spearman
FORCE_INTERP = False         # True to recompute from scratch

_vc = adata_interp.obs[INTERP_CT_KEY].value_counts()
_n_samp = adata_interp.obs.groupby(INTERP_CT_KEY, observed=True)[SAMPLE_KEY].nunique()
_eligible = [ct for ct in _vc.index if _n_samp.get(ct, 0) >= MIN_SAMPLES]
INTERP_CELL_TYPES = _eligible if N_INTERP_CT is None else _eligible[:N_INTERP_CT]

print(f"{len(_eligible)}/{len(_vc)} cell types have >= {MIN_SAMPLES} samples; "
      f"running on {len(INTERP_CELL_TYPES)}")
print("dropped for too few samples:", [ct for ct in _vc.index if ct not in _eligible][:8], "...")

interp_dict = run_interpretation_pipeline(
    adata_interp,
    cell_type_key=INTERP_CT_KEY,
    sample_key=SAMPLE_KEY,
    output_dir=INTERP_DIR,
    cell_types=INTERP_CELL_TYPES,
    weights_key="X_cell_weights",
    min_expr_frac=0.05,
    min_head_activity_frac=0.1,
    corr_method="spearman",
    n_top_genes=None,            # rank ALL genes
    save_expression=True,        # needed to re-correlate genes vs covariate later
    force=FORCE_INTERP,
)
print(f"\n{len(interp_dict)} cell types aggregated -> {INTERP_DIR}")

### Which (cell type, head) pairs track the covariate?

Spearman r between each head's attention and the covariate, averaged within patient first; stars mark BH-FDR < 0.05.

In [ ]:
# attention vs covariate, per cell type x head
from scipy.stats import spearmanr

# average within patient for a patient-level covariate, never for Inflammation_score, which varies within patient
GROUP_BY_PATIENT = COVARIATE != "Inflammation_score"   # False -> per-sample

def sample_covariate(col=COVARIATE, positive=COV_POSITIVE):
    """One numeric value per sample, indexed by sample ID. Categorical -> positive==1."""
    s = meta_adata.obs[col]
    return (s == positive).astype(float).where(s.notna()) if s.dtype == object or str(s.dtype) == "category" \
        else s.astype(float)

def head_covariate_corr(interp_dict, values, group=None):
    """Spearman r(head attention, covariate) per cell type x head; `group` averages within group first."""
    r_rows, p_rows = [], []
    for ct, interp in interp_dict.items():
        attn = interp.attn_filtered
        shared = attn.index.intersection(values.dropna().index)
        a, v = attn.loc[shared], values.loc[shared]
        if group is not None:
            g = group.loc[shared]
            a = a.groupby(g).mean()
            v = v.groupby(g).first().loc[a.index]
        r_row, p_row = {"cell_type": ct}, {"cell_type": ct}
        for head in interp.active_heads:
            x, y = v.to_numpy(float), a[head].to_numpy(float)
            ok = np.isfinite(x) & np.isfinite(y)
            if ok.sum() < 5 or np.unique(y[ok]).size < 2:
                r_row[head] = p_row[head] = np.nan
            else:
                r, p = spearmanr(x[ok], y[ok])
                r_row[head], p_row[head] = r, p
        r_rows.append(r_row); p_rows.append(p_row)
    r_df = pd.DataFrame(r_rows).set_index("cell_type")
    p_df = pd.DataFrame(p_rows).set_index("cell_type")
    cols = sorted(r_df.columns, key=lambda h: int(h.split("_")[1]))
    return r_df[cols], p_df[cols]

_vals  = sample_covariate()
_group = meta_adata.obs[PATIENT_KEY].astype(str) if GROUP_BY_PATIENT else None
r_df, p_df = head_covariate_corr(interp_dict, _vals, group=_group)
q_df = correct_pvalues_matrix(p_df, method="fdr_bh")

n_unit = _group.loc[_vals.dropna().index].nunique() if GROUP_BY_PATIENT else int(_vals.notna().sum())
print(f"covariate={COVARIATE} ({COV_POSITIVE}=1) | unit = "
      f"{'patient' if GROUP_BY_PATIENT else 'sample'} | n = {n_unit}")
print(f"FDR<0.05 in {int((q_df < 0.05).to_numpy().sum())} of "
      f"{int(q_df.notna().to_numpy().sum())} (cell type, head) pairs")
r_df.round(2)

In [ ]:
%matplotlib inline
# heatmap: attention-covariate correlation per cell type x head
_order = r_df.abs().max(axis=1).sort_values(ascending=False).index
_r, _q = r_df.loc[_order], q_df.loc[_order]
_annot = _q.map(lambda v: "*" if pd.notna(v) and v < 0.05 else "")
_vmax = np.nanmax(np.abs(_r.to_numpy())) or 1.0

fig, ax = plt.subplots(figsize=(1.0 * _r.shape[1] + 4.5, 0.34 * len(_r) + 2.2), dpi=130)
sns.heatmap(_r, ax=ax, cmap="RdBu_r", vmin=-_vmax, vmax=_vmax, center=0,
            annot=_annot, fmt="s", annot_kws={"size": 15, "color": "black"},
            linewidths=0.4, linecolor="white", cbar_kws={"label": "Spearman r"})
ax.set_xlabel(""); ax.set_ylabel("")
ax.set_title(f"{INTERP_REP}\nattention vs {COVARIATE} "
             f"({'patient' if GROUP_BY_PATIENT else 'sample'}-level, n={n_unit}; * FDR<0.05)\n"
             f"red = more attention in {COV_POSITIVE}",
             loc="left", fontsize=15)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=13)
plt.setp(ax.get_xticklabels(), rotation=0, fontsize=13)
plt.tight_layout()
plt.savefig('hca-gut-atlas-tutorial/images/temp_images/attention_overview.svg')
plt.show()

# the ranked table is the thing to actually read off
_long = (r_df.stack().rename("r").to_frame()
         .join(q_df.stack().rename("q_fdr"))
         .reset_index().rename(columns={"level_1": "head"}))
_long["abs_r"] = _long["r"].abs()
print("top (cell type, head) pairs by |r|:")
print(_long.sort_values("abs_r", ascending=False).head(12).round(3).to_string(index=False))

### Genes associated with a head, inside a cell type

Genes whose per-sample mean expression tracks a head's attention within one cell type -- a ranking device, not a significance test.

In [ ]:
%matplotlib inline
# top cell types x their strongest heads, ranked by |r(attention, covariate)|: gene rankings
import re
from IPython.display import SVG, display

N_CELL_TYPES = 20
N_HEADS_PER_CT = 2
N_TOP_GENES = 20
SHOW_INLINE = True                      # plot_head_top_genes closes the fig when saving
FIG_DIR = Path("hca-gut-atlas-tutorial/images/temp_images")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# rank cell types the same way the heatmap orders them
RANK_BY = r_df.abs().max(axis=1)        # strongest head-covariate coupling in that cell type
top_cts = RANK_BY.sort_values(ascending=False).head(N_CELL_TYPES).index.tolist()

def _slug(s):
    """Make a cell type name safe to use in a filename."""
    return re.sub(r"[^0-9A-Za-z._-]+", "_", str(s)).strip("_")

summary = []
for rank, ct in enumerate(top_cts, 1):
    # heads active for THIS cell type, strongest |r| first (NaN = head was inactive here)
    heads = r_df.loc[ct].dropna().abs().sort_values(ascending=False).head(N_HEADS_PER_CT).index.tolist()
    if not heads:
        print(f"[{rank:2d}] {ct}: no active heads, skipped")
        continue

    res = correlate_and_rank(interp_dict[ct], corr_method="spearman",
                             n_top_genes=N_TOP_GENES, verbose=False)
    heads = [h for h in heads if h in res["gene_rankings"]]   # active_heads can differ again
    if not heads:
        print(f"[{rank:2d}] {ct}: heads dropped by correlate_and_rank, skipped")
        continue

    tag = "  |  ".join(f"{h} r={r_df.loc[ct, h]:+.2f} q={q_df.loc[ct, h]:.2g}" for h in heads)
    print(f"\n[{rank:2d}] {ct}  ({tag})")

    out = FIG_DIR / f"{_slug(ct)}_{'_'.join(heads)}.svg"
    plot_head_top_genes(res, heads=heads, figsize_per_head=(3, 4), save_path=out)
    if SHOW_INLINE:
        display(SVG(filename=str(out)))   # save_path -> plt.close(), so re-display the file

    for h in heads:
        rk = res["gene_rankings"][h].head(N_TOP_GENES)
        print(f"  top {N_TOP_GENES} genes, {h}: {', '.join(rk['gene'].astype(str))}")
        summary.append(rk.assign(cell_type=ct, head=h, ct_rank=rank,
                                 r_attention_covariate=r_df.loc[ct, h],
                                 q_fdr_covariate=q_df.loc[ct, h]))

gene_summary = pd.concat(summary, ignore_index=True)
gene_summary.to_csv(INTERP_DIR / f"top{N_CELL_TYPES}ct_top{N_HEADS_PER_CT}heads_genes.csv", index=False)
print(f"\n{gene_summary['cell_type'].nunique()} cell types x "
      f"{gene_summary.groupby('cell_type')['head'].nunique().max()} heads -> "
      f"{len(gene_summary)} gene rows  ->  {INTERP_DIR}")
gene_summary.head(20).round(3)

In [ ]:
%matplotlib inline
# gene rankings for one (cell type, head); defaults to the strongest pair from the heatmap
_best = _long.sort_values("abs_r", ascending=False).iloc[0]
FOCUS_CT   = _best["cell_type"]
FOCUS_HEAD = _best["head"]
N_TOP_GENES = 20


print(f"focus: {FOCUS_CT}  |  {FOCUS_HEAD}  "
      f"(r={_best['r']:.2f}, FDR q={_best['q_fdr']:.3g})")

_res = correlate_and_rank(interp_dict[FOCUS_CT], corr_method="spearman",
                          n_top_genes=N_TOP_GENES, verbose=False)
plot_head_top_genes(
    _res, heads=[FOCUS_HEAD],
    figsize_per_head=(3, 4),
    save_path=f'hca-gut-atlas-tutorial/images/temp_images/{FOCUS_CT}_{FOCUS_HEAD}.svg',
    )

_rank = _res["gene_rankings"][FOCUS_HEAD]
print(f"\ntop {N_TOP_GENES} genes for {FOCUS_CT} / {FOCUS_HEAD}:")
print(_rank.head(N_TOP_GENES).round(3).to_string(index=False))

In [ ]:
%matplotlib inline
# gene rankings for one (cell type, head); defaults to the strongest pair from the heatmap
_best = _long.sort_values("abs_r", ascending=False).iloc[0]
FOCUS_CT   = "Classical Monocytes"
FOCUS_HEAD = _best["head"]
N_TOP_GENES = 20

print(f"focus: {FOCUS_CT}  |  {FOCUS_HEAD}  "
      f"(r={_best['r']:.2f}, FDR q={_best['q_fdr']:.3g})")

_res = correlate_and_rank(interp_dict[FOCUS_CT], corr_method="spearman",
                          n_top_genes=N_TOP_GENES, verbose=False)
plot_head_top_genes(_res, heads=[FOCUS_HEAD], figsize_per_head=(6, 5))

_rank = _res["gene_rankings"][FOCUS_HEAD]
print(f"\ntop {N_TOP_GENES} genes for {FOCUS_CT} / {FOCUS_HEAD}:")
print(_rank.head(N_TOP_GENES).round(3).to_string(index=False))

In [ ]:
%matplotlib inline
# sanity-check a single gene, and close the loop back to the covariate
FOCUS_GENE = _rank["gene"].iloc[0]
plot_gene_head_scatter(interp_dict[FOCUS_CT], _res, gene=FOCUS_GENE, head=FOCUS_HEAD)

# re-correlate each top gene against the covariate itself, at the same level as the heatmap
_interp = interp_dict[FOCUS_CT]
_vals_ct = _vals.reindex(_interp.sample_expression.index)
_expr = _interp.sample_expression
if GROUP_BY_PATIENT:
    _g = meta_adata.obs[PATIENT_KEY].astype(str).reindex(_expr.index)
    _expr, _vals_ct = _expr.groupby(_g).mean(), _vals_ct.groupby(_g).first()
_ok = _vals_ct.notna().to_numpy()

rows = []
for g in _rank["gene"].head(N_TOP_GENES):
    if g not in _expr.columns:
        continue
    e = _expr[g].to_numpy(float)[_ok]
    if np.unique(e).size < 2:
        continue
    r_cov, p_cov = spearmanr(_vals_ct.to_numpy(float)[_ok], e)
    rows.append({"gene": g,
                 "r_attention": float(_res["correlations"].loc[g, FOCUS_HEAD]),
                 f"r_{COVARIATE}": r_cov, "p": p_cov})
gene_cov = pd.DataFrame(rows)
gene_cov["q_fdr"] = correct_pvalues_matrix(gene_cov[["p"]], method="fdr_bh")["p"].to_numpy()

print(f"{FOCUS_CT} / {FOCUS_HEAD}: top genes re-correlated against {COVARIATE} "
      f"({'patient' if GROUP_BY_PATIENT else 'sample'}-level, n={int(_ok.sum())})")
print(gene_cov.sort_values("q_fdr").round(3).to_string(index=False))
print(f"\n{int((gene_cov['q_fdr'] < 0.05).sum())}/{len(gene_cov)} of the head's top genes are "
      f"themselves FDR<0.05 for {COVARIATE}")

In [ ]:
# do the heads' top genes track the covariate themselves?
from scipy.stats import spearmanr

assert "gene_summary" in globals(), "run the top-20 cell-type / top-2-head loop first"
VERBOSE_BLOCKS = "hits"        # "hits" | "all" | "none"

_vals = sample_covariate()
_pat  = meta_adata.obs[PATIENT_KEY].astype(str)
R_COV = f"r_{COVARIATE}"

def covariate_correlations(ct, genes):
    """Spearman r(gene expression, covariate) inside one cell type, at the unit used above."""
    interp = interp_dict[ct]
    expr   = interp.sample_expression
    vals   = _vals.reindex(expr.index)
    if GROUP_BY_PATIENT:
        g = _pat.reindex(expr.index)
        expr, vals = expr.groupby(g).mean(), vals.groupby(g).first()
    ok = vals.notna().to_numpy()
    x  = vals.to_numpy(float)[ok]
    out = []
    for gene in genes:
        if gene not in expr.columns:
            continue
        e = expr[gene].to_numpy(float)[ok]
        if np.unique(e).size < 2:
            continue
        r, p = spearmanr(x, e)
        out.append({"gene": gene, R_COV: r, "p": p, "n": int(ok.sum())})
    return pd.DataFrame(out)

rows = []
for (ct, head), grp in gene_summary.groupby(["cell_type", "head"], sort=False):
    res = covariate_correlations(ct, grp["gene"].tolist())
    if res.empty:
        print(f"skip {ct} / {head}: no usable genes")
        continue
    rows.append(res.assign(
        cell_type=ct, head=head, ct_rank=grp["ct_rank"].iat[0],
        r_attention=grp.set_index("gene")["correlation"].reindex(res["gene"]).to_numpy(),
        r_attention_covariate=grp["r_attention_covariate"].iat[0],
        q_fdr_attention=grp["q_fdr_covariate"].iat[0]))
gene_cov_all = pd.concat(rows, ignore_index=True)

# one BH correction over the whole sweep: a gene surfaced by two heads is still a single test
tests = gene_cov_all.drop_duplicates(["cell_type", "gene"])[["cell_type", "gene", "p"]].copy()
tests["q_fdr"] = correct_pvalues_matrix(tests[["p"]], method="fdr_bh")["p"].to_numpy()
gene_cov_all = gene_cov_all.merge(tests[["cell_type", "gene", "q_fdr"]],
                                  on=["cell_type", "gene"], how="left")

# does the attention chain predict the sign of gene->covariate?
_expected = np.sign(gene_cov_all["r_attention_covariate"] * gene_cov_all["r_attention"])
gene_cov_all["sign_as_predicted"] = _expected == np.sign(gene_cov_all[R_COV])

unit = "patient" if GROUP_BY_PATIENT else "sample"
print(f"{gene_cov_all['cell_type'].nunique()} cell types x "
      f"{gene_cov_all.groupby('cell_type')['head'].nunique().max()} heads | "
      f"{len(tests)} unique gene tests vs {COVARIATE} ({unit}-level, n={gene_cov_all['n'].iat[0]}) | "
      f"BH-FDR over all {len(tests)}")

COLS = ["gene", "r_attention", R_COV, "p", "q_fdr", "sign_as_predicted"]
for (ct, head), grp in gene_cov_all.groupby(["cell_type", "head"], sort=False):
    n_hit = int((grp["q_fdr"] < 0.05).sum())
    if VERBOSE_BLOCKS == "none" or (VERBOSE_BLOCKS == "hits" and n_hit == 0):
        continue
    print(f"\n[{grp['ct_rank'].iat[0]:2d}] {ct} / {head}  "
          f"(attention vs {COVARIATE}: r={grp['r_attention_covariate'].iat[0]:+.2f}, "
          f"q={grp['q_fdr_attention'].iat[0]:.3g})")
    print(grp[COLS].sort_values("q_fdr").round(3).to_string(index=False))
    print(f"  {n_hit}/{len(grp)} of the head's top genes are themselves FDR<0.05 for {COVARIATE}")

# per (cell type, head) summary
per_pair = (gene_cov_all.groupby(["ct_rank", "cell_type", "head"])
            .agg(n_genes=("gene", "size"),
                 n_fdr05=("q_fdr", lambda s: int((s < 0.05).sum())),
                 max_abs_r=(R_COV, lambda s: s.abs().max()),
                 frac_sign_ok=("sign_as_predicted", "mean"),
                 r_attn_cov=("r_attention_covariate", "first"))
            .reset_index().sort_values(["n_fdr05", "max_abs_r"], ascending=False))
print(f"\n=== per (cell type, head): genes that survive against {COVARIATE} ===")
print(per_pair.round(3).to_string(index=False))

_sig = gene_cov_all[gene_cov_all["q_fdr"] < 0.05]
print(f"\n{len(_sig)} of {len(tests)} unique tests reach FDR<0.05 "
      f"({_sig['cell_type'].nunique()} cell types); "
      f"{_sig['sign_as_predicted'].mean():.0%} of those have the sign the attention chain predicts")
print("\ntop 20 by |r| against the covariate:")
print(_sig.reindex(_sig[R_COV].abs().sort_values(ascending=False).index)
      .head(20)[["cell_type", "head"] + COLS].round(3).to_string(index=False))

gene_cov_all.to_csv(INTERP_DIR / f"top20ct_genes_vs_{COVARIATE}.csv", index=False)
print(f"\nsaved -> {INTERP_DIR / f'top20ct_genes_vs_{COVARIATE}.csv'}")

## Reload Gene correlation results

In [ ]:
COVARIATE = "Remission_clean"
INTERP_REP = "sclr_remission_batchaware_seed0" 
SCLR_DIR   = Path("hca-gut-atlas-downstream/data/taurus_data/sampleclr_beyond_batch")
INTERP_DIR = SCLR_DIR / "interpretation" / INTERP_REP

In [ ]:
gene_cov_all = pd.read_csv(INTERP_DIR / f"top20ct_genes_vs_{COVARIATE}.csv")

In [ ]:
gene_cov_all

In [ ]:
gene_cov_all['cell_type'].unique()

In [ ]:
REMISSION_COLOR = "#2166AC"      # blue  = up in remission
NONREMISSION_COLOR = "#B2182B"   # red   = up in non-remission (CVD-safe pair)


def plot_remission_association(df, cell_type, head=None, top_n=20,
                               fdr_thresh=0.05, value_col="r_Remission_clean",
                               ax=None, save_path=None):
    """Diverging bar chart of the top genes by |association| for one cell type.

    `head=None` picks the head with the most FDR<`fdr_thresh` genes; those genes get a star.
    """
    sub = df[df["cell_type"] == cell_type].copy()
    if sub.empty:
        raise ValueError(f"no rows for cell_type={cell_type!r}; "
                         f"available: {sorted(df['cell_type'].unique())}")
    if head is None:
        head = (sub.assign(_sig=sub["q_fdr"] < fdr_thresh)
                   .groupby("head")["_sig"].sum().idxmax())
    sub = sub[sub["head"] == head]
    if sub.empty:
        raise ValueError(f"no rows for cell_type={cell_type!r}, head={head!r}")

    # top_n by absolute association, then ordered for display
    sub = sub.reindex(sub[value_col].abs().sort_values(ascending=False).index)
    sub = sub.head(top_n).sort_values(value_col)

    genes = sub["gene"].tolist()
    vals = sub[value_col].to_numpy()
    sig = (sub["q_fdr"] < fdr_thresh).to_numpy()
    n_pat = int(sub["n"].iloc[0]) if "n" in sub.columns else None
    y = np.arange(len(genes))

    if ax is None:
        fig, ax = plt.subplots(figsize=(3.5, 0.17 * len(genes) + 1.3))
    else:
        fig = ax.figure

    ax.barh(y, vals, height=0.72, edgecolor="none", zorder=2,
            color=[REMISSION_COLOR if v > 0 else NONREMISSION_COLOR for v in vals])
    ax.axvline(0, color="0.2", lw=0.9, zorder=4)
    for yi, v, s in zip(y, vals, sig):          # FDR hits: star just past the bar tip
        if s:
            off = 0.03 if v >= 0 else -0.03
            ax.text(v + off, yi-0.15, "*", color="black",
                    fontsize=8, fontweight="bold", va="center",
                    ha="left" if v >= 0 else "right", zorder=5)

    ax.set_yticks(y)
    ax.set_yticklabels(genes, fontsize=7)       # plain labels; significance shown by *
    ax.set_ylim(-0.7, len(genes) - 0.3)
    ax.set_xlim(-1.0, 1.0)
    ax.set_xticks([-0.8, -0.4, 0, 0.4, 0.8])
    ax.set_title(f"{cell_type}", fontsize=9)
    ax.set_xlabel("Spearman r vs remission", fontsize=8)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.margins(y=0.01)

    
    if n_pat is not None:
        ax.text(0.5, -0.5, f"n = {n_pat} patients \u00b7 * = FDR<{fdr_thresh}",
                transform=ax.transAxes, ha="center", va="top",
                fontsize=6.5, color="0.4")

    if save_path:
        fig.tight_layout()
        fig.savefig(save_path, dpi=300, bbox_inches="tight", transparent=True)
    return fig, ax

In [ ]:
plot_remission_association(gene_cov_all, 'M0 Macrophages', save_path='hca-gut-atlas-tutorial/images/temp_images/M0_Macrophages_remission_association.svg')

In [ ]:
plot_remission_association(gene_cov_all, 'Mid Crypt Colonocytes', save_path='hca-gut-atlas-tutorial/images/temp_images/Mid_Crypt_Colonocytes_remission_association.svg')

In [ ]:
plot_remission_association(gene_cov_all, 'Goblet Cells', save_path='hca-gut-atlas-tutorial/images/temp_images/Goblet_Cells_remission_association.svg')